# Muse-Glimmer-30B v1.0.0 — Kaggle T4 x2 Production Demo / Demo Production trên Kaggle T4 x2

> **English:** Canonical operator notebook for the frozen public `v1.0.0` release.
> 
> **Tiếng Việt:** Notebook vận hành canonical cho public release `v1.0.0` đã khóa.

## English

This notebook does **not** reimplement the runtime. It clones and operates the published repository at the exact accepted tag/commit/tree, then uses the repository's own `readiness.sh` / `external.sh` contract.

### Notebook goals

* validate the Kaggle **NVIDIA T4 x2** hardware profile;
* verify the required attached model/runtime inputs;
* pin the exact public `v1.0.0` source identity before GPU use;
* use a secret-only Bearer token (`MUSE_API_TOKEN`);
* start the canonical DFlash2 backend and authenticated gateway;
* expose the gateway through an optional Cloudflare Quick Tunnel;
* demonstrate authentication, `/ready`, `/v1/models`, non-streaming chat, and SSE streaming;
* write a small **secret-free** session evidence JSON;
* stop the tunnel, gateway, and backend at the end by default.

This is a **production-style demo**, not a hosted service, SLA, HA deployment, or multi-tenant platform.

---

## Tiếng Việt

Notebook này **không triển khai lại runtime**. Notebook clone và vận hành repository đã được publish tại chính xác tag/commit/tree đã được chấp nhận, sau đó sử dụng contract `readiness.sh` / `external.sh` của chính repository.

### Mục tiêu của notebook

* xác minh cấu hình phần cứng Kaggle **NVIDIA T4 x2**;
* xác minh các model/runtime input bắt buộc đã được attach;
* khóa chính xác source identity của public `v1.0.0` trước khi sử dụng GPU;
* sử dụng Bearer token chỉ lấy từ secret (`MUSE_API_TOKEN`);
* khởi động DFlash2 backend canonical và gateway có xác thực;
* expose gateway thông qua Cloudflare Quick Tunnel tùy chọn;
* minh họa authentication, `/ready`, `/v1/models`, non-streaming chat và SSE streaming;
* ghi một session evidence JSON nhỏ và **không chứa secret**;
* mặc định dừng tunnel, gateway và backend khi notebook hoàn tất.

Đây là một **production-style demo**, không phải dịch vụ được host thường trực, không cung cấp SLA, không phải triển khai HA và không phải nền tảng multi-tenant.


## Before you run / Trước khi chạy

### English

In Kaggle Notebook settings:

1. **Accelerator:** select **GPU T4 x2**.
2. **Internet:** enable Internet for the Git clone and Cloudflare Quick Tunnel.
3. **Attach the exact three Kaggle resources below.** Do not search by filename alone and do not choose a different model variation.

   **A. Target model — Kaggle Model**
   - Publisher/resource: `dangkhoa2016/bartowski-muse-glimmer-30b-gguf`
   - Framework / variation / version: **GGUF / `q4-k-m` / `1`**
   - Direct link: https://www.kaggle.com/models/dangkhoa2016/bartowski-muse-glimmer-30b-gguf/Gguf/q4-k-m/1
   - Required file: `Muse-Glimmer-30B-Q4_K_M.gguf`
   - Expected SHA-256: `0d3fc85f61d10fdc84072f0bba6005d61c1ac5605a2627b0fd5ea4ff8194c384`

   **B. DFlash2 draft — Kaggle Model**
   - Publisher/resource: `dangkhoa2016/incoai-muse-glimmer-30b-dflash2-gguf`
   - Framework / variation / version: **GGUF / `default` / `1`**
   - Direct link: https://www.kaggle.com/models/dangkhoa2016/incoai-muse-glimmer-30b-dflash2-gguf/Gguf/default/1
   - Required file: `Muse-Glimmer-30B-DFlash2-Q4_K_M.gguf`
   - Expected SHA-256: `93dbfb6f88e4645dec1347cf93f9d6fc80b90d413038722385b2a8e53565c949`
   - The Kaggle variation is named `default`; the exact file inside that variation is the qualified Q4_K_M DFlash2 draft.

   **C. Pinned llama.cpp runtime — Kaggle Dataset**
   - Publisher/resource: `dangkhoa2016/muse-glimmer-30b-dflash2-llama-runtime`
   - Direct link: https://www.kaggle.com/datasets/dangkhoa2016/muse-glimmer-30b-dflash2-llama-runtime
   - This is a **Dataset**, not a Kaggle Model.

   In the Kaggle notebook UI, use **Add Input**, open each resource above, and attach the specified model variation/version or dataset. Kaggle may mount Models under `/kaggle/input/models/...` and Datasets under `/kaggle/input/datasets/...`; this notebook discovers both recursively.

4. **Recommended:** add a Kaggle Secret named **`MUSE_API_TOKEN`** containing at least 32 printable characters.

   - If the secret is missing, the notebook automatically creates a cryptographically secure temporary token.
   - The generated token is stored at `/kaggle/working/.muse-secrets/MUSE_API_TOKEN` with file mode `0600`.
   - The token value is **never printed, hashed, or written into the evidence JSON**.
   - The notebook prints a warning containing only the temporary credential file path.
   - If you intentionally need the generated token for an external client, read the file privately. Do not publish that cell output or include the secret file in Notebook Outputs.

### Expected Kaggle mount examples

The final qualified T4 x2 run resolved the three inputs under paths shaped like:

```text
/kaggle/input/models/dangkhoa2016/bartowski-muse-glimmer-30b-gguf/gguf/q4-k-m/1/Muse-Glimmer-30B-Q4_K_M.gguf
/kaggle/input/models/dangkhoa2016/incoai-muse-glimmer-30b-dflash2-gguf/gguf/default/1/Muse-Glimmer-30B-DFlash2-Q4_K_M.gguf
/kaggle/input/datasets/dangkhoa2016/muse-glimmer-30b-dflash2-llama-runtime
```

The notebook does not require those roots to be direct children of `/kaggle/input`; recursive discovery is intentional.

### Default safety behavior

* `AUTO_CLEANUP_AT_END=True` stops the Quick Tunnel, authenticated gateway, and backend when the demo finishes.
* `AUTO_DELETE_GENERATED_TOKEN_AT_END=True` deletes the notebook-generated token file during normal cleanup.

Set either option to `False` only when you intentionally need to keep the temporary demo running and understand the exposure implications.

### Production notebook behavior

This notebook uses the frozen public `v1.0.0` repository as its runtime source and adds notebook-owned operational safeguards without modifying that source tree.

* Safe same-session recovery stops only processes matching exact Muse demo signatures and refuses to kill unknown listeners.
* The local production core is qualified first through the authenticated loopback gateway.
* Public Cloudflare Quick Tunnel transport is qualified separately so transient public transport problems do not invalidate a healthy local model/runtime.
* Real non-streaming and SSE requests use `chat_template_kwargs.reasoning_strength=low` together with the public `max_tokens=512` request cap.
* The notebook-owned auth-proxy launcher applies `--backend-timeout 120` while preserving the frozen repository source manifest.
* Secret fallback uses a temporary `MUSE_API_TOKEN` file with mode `0600` and deletes it during normal cleanup.
* Forensic artifacts are created only when the current run captures a failure.

---

### Tiếng Việt

Trong phần cài đặt Kaggle Notebook:

1. **Accelerator:** chọn **GPU T4 x2**.
2. **Internet:** bật Internet để notebook có thể clone Git repository và sử dụng Cloudflare Quick Tunnel.
3. **Đính kèm chính xác ba Kaggle resource dưới đây.** Không chỉ tìm theo tên file và không chọn variation khác.

   **A. Target model — Kaggle Model**
   - Publisher/resource: `dangkhoa2016/bartowski-muse-glimmer-30b-gguf`
   - Framework / variation / version: **GGUF / `q4-k-m` / `1`**
   - Link trực tiếp: https://www.kaggle.com/models/dangkhoa2016/bartowski-muse-glimmer-30b-gguf/Gguf/q4-k-m/1
   - File bắt buộc: `Muse-Glimmer-30B-Q4_K_M.gguf`
   - SHA-256 mong đợi: `0d3fc85f61d10fdc84072f0bba6005d61c1ac5605a2627b0fd5ea4ff8194c384`

   **B. DFlash2 draft — Kaggle Model**
   - Publisher/resource: `dangkhoa2016/incoai-muse-glimmer-30b-dflash2-gguf`
   - Framework / variation / version: **GGUF / `default` / `1`**
   - Link trực tiếp: https://www.kaggle.com/models/dangkhoa2016/incoai-muse-glimmer-30b-dflash2-gguf/Gguf/default/1
   - File bắt buộc: `Muse-Glimmer-30B-DFlash2-Q4_K_M.gguf`
   - SHA-256 mong đợi: `93dbfb6f88e4645dec1347cf93f9d6fc80b90d413038722385b2a8e53565c949`
   - Variation trên Kaggle có tên `default`; file chính xác bên trong variation này là DFlash2 draft Q4_K_M đã được qualification.

   **C. Pinned llama.cpp runtime — Kaggle Dataset**
   - Publisher/resource: `dangkhoa2016/muse-glimmer-30b-dflash2-llama-runtime`
   - Link trực tiếp: https://www.kaggle.com/datasets/dangkhoa2016/muse-glimmer-30b-dflash2-llama-runtime
   - Đây là **Dataset**, không phải Kaggle Model.

   Trong giao diện Kaggle notebook, dùng **Add Input**, mở từng resource ở trên rồi attach đúng model variation/version hoặc dataset đã chỉ định. Kaggle có thể mount Models dưới `/kaggle/input/models/...` và Datasets dưới `/kaggle/input/datasets/...`; notebook này tự động tìm đệ quy cả hai loại.

4. **Khuyến nghị:** tạo một Kaggle Secret có tên **`MUSE_API_TOKEN`** với ít nhất 32 ký tự có thể in được.

   - Nếu secret chưa được cấu hình, notebook sẽ tự động tạo một token tạm thời bằng cơ chế sinh số ngẫu nhiên mật mã an toàn.
   - Token được tạo sẽ được lưu tại `/kaggle/working/.muse-secrets/MUSE_API_TOKEN` với file mode `0600`.
   - Giá trị token **không bao giờ được in ra, hash hoặc ghi vào evidence JSON**.
   - Notebook chỉ in cảnh báo kèm đường dẫn tới file credential tạm thời.
   - Nếu bạn chủ động cần token được tạo để sử dụng từ một external client, hãy đọc file đó một cách riêng tư. Không publish output của cell chứa secret và không đưa file secret vào Notebook Outputs.

### Ví dụ đường dẫn mount trên Kaggle

Final T4 x2 run đã qualification thành công resolve ba input theo dạng:

```text
/kaggle/input/models/dangkhoa2016/bartowski-muse-glimmer-30b-gguf/gguf/q4-k-m/1/Muse-Glimmer-30B-Q4_K_M.gguf
/kaggle/input/models/dangkhoa2016/incoai-muse-glimmer-30b-dflash2-gguf/gguf/default/1/Muse-Glimmer-30B-DFlash2-Q4_K_M.gguf
/kaggle/input/datasets/dangkhoa2016/muse-glimmer-30b-dflash2-llama-runtime
```

Notebook không yêu cầu các root trên phải là direct child của `/kaggle/input`; việc tìm đệ quy là chủ ý.

### Hành vi an toàn mặc định

* `AUTO_CLEANUP_AT_END=True` sẽ dừng Quick Tunnel, authenticated gateway và backend khi demo kết thúc.
* `AUTO_DELETE_GENERATED_TOKEN_AT_END=True` sẽ xóa file token do notebook tự tạo trong quá trình cleanup bình thường.

Chỉ đặt một trong hai tùy chọn trên thành `False` khi bạn chủ động muốn giữ demo tạm thời tiếp tục chạy và hiểu rõ các rủi ro liên quan đến việc expose dịch vụ.

### Hành vi của production notebook

Notebook sử dụng repository public `v1.0.0` đã khóa làm runtime source và bổ sung các safeguard vận hành do notebook quản lý mà không thay đổi source tree đó.

* Cơ chế recovery trong cùng session chỉ dừng những process khớp chính xác với Muse demo signature và từ chối kill các listener không xác định.
* Local production core được qualification trước thông qua authenticated loopback gateway.
* Public transport qua Cloudflare Quick Tunnel được qualification ở phase riêng để lỗi public transport tạm thời không làm mất hiệu lực của một local model/runtime đang hoạt động đúng.
* Các request non-streaming và SSE thực tế sử dụng `chat_template_kwargs.reasoning_strength=low` cùng public request cap `max_tokens=512`.
* Auth-proxy launcher do notebook quản lý áp dụng `--backend-timeout 120` trong khi vẫn giữ nguyên frozen repository source manifest.
* Khi thiếu Kaggle Secret, notebook sử dụng file `MUSE_API_TOKEN` tạm thời với mode `0600` và xóa file này trong quá trình cleanup bình thường.
* Forensic artifact chỉ được tạo khi chính run hiện tại ghi nhận một failure.


## 1. Configure the canonical production notebook contract / Cấu hình contract canonical của notebook production

**English:** This cell defines the frozen public release identity, Kaggle working paths, safety defaults, helper functions, canonical serving expectations, the production request policy (`reasoning_strength=low`), and the notebook-owned 120-second auth-proxy timeout overlay. It does not start the model. Expected output begins with `CONFIG=READY` and prints only non-secret configuration values.

**Tiếng Việt:** Cell này khai báo danh tính public release đã khóa, các đường dẫn làm việc trên Kaggle, mặc định an toàn, các hàm hỗ trợ, cấu hình serving canonical, chính sách request của production (`reasoning_strength=low`) và lớp timeout 120 giây do notebook quản lý cho auth proxy. Cell này chưa khởi động model. Output mong đợi bắt đầu bằng `CONFIG=READY` và chỉ in các giá trị cấu hình không nhạy cảm.


In [ ]:
from pathlib import Path
import os, json, subprocess, shutil, hashlib, datetime, getpass, urllib.parse, secrets, signal, socket, time

REPO = "dangkhoa2016/Muse-Glimmer-30B-GGUF-DFlash2-Kaggle-GPU-T4x2"
REPO_URL = f"https://github.com/{REPO}.git"
TAG = "v1.0.0"

EXPECTED_HEAD = "a54e66542e0cfdd168458530bc19f61c333ecd33"
EXPECTED_TREE = "ca2b545667471aa1f4078409d13d32bbe8f04b7e"

WORKDIR = Path("/kaggle/working/muse-glimmer-30b-v1.0.0")
INPUT_ROOT = Path("/kaggle/input")
BIN_DIR = Path("/kaggle/working/.muse-demo-bin")
PYTHON_SHIM_DIR = BIN_DIR / "python-shim"
EVIDENCE_PATH = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-demo-evidence.json")
GENERATED_SECRET_DIR = Path("/kaggle/working/.muse-secrets")
GENERATED_TOKEN_FILE = GENERATED_SECRET_DIR / "MUSE_API_TOKEN"
FORENSIC_DIR = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-forensic")
FORENSIC_ZIP_PATH = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-forensic.zip")

RUN_NONSTREAM_DEMO = True
RUN_SSE_DEMO = True
AUTO_CLEANUP_AT_END = True
AUTO_DELETE_GENERATED_TOKEN_AT_END = True
PUBLIC_DNS_RETRY_SECONDS = 90
PUBLIC_RETRY_INTERVAL_SECONDS = 2
PUBLIC_HTTP_TIMEOUT_SECONDS = 12
DEMO_REQUEST_MAX_TOKENS = 512
DEMO_REASONING_STRENGTH = "low"
AUTH_PROXY_BACKEND_TIMEOUT_SECONDS = 120

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["SERVE_PROFILE"] = "dflash2"
os.environ["EXPOSURE_MODE"] = "quick"
os.environ["PYTHONUNBUFFERED"] = "1"

SESSION = {
    "notebook_contract": "Muse-Glimmer-30B v1.0.0 Kaggle production demo",
    "tag": TAG,
    "expected_head": EXPECTED_HEAD,
    "expected_tree": EXPECTED_TREE,
    "started_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "forensic_capture_count": 0,
}

def run(args, *, cwd=None, env=None, capture=True, check=True):
    if isinstance(args, str):
        raise TypeError("run() requires a list/tuple, not a shell string")
    p = subprocess.run(
        list(args),
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
        check=False,
    )
    if capture and p.stdout:
        print(p.stdout.rstrip())
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed rc={p.returncode}: {args!r}")
    return p




def classify_muse_demo_process(cmdline):
    """Return a narrow Muse demo process class, or None for anything ambiguous."""
    cmdline = (cmdline or "").strip()
    if not cmdline:
        return None

    canonical_cloudflared = str(BIN_DIR / "cloudflared")

    if (
        cmdline.startswith(canonical_cloudflared + " ")
        and " tunnel " in f" {cmdline} "
        and "--url" in cmdline
        and "http://127.0.0.1:8090" in cmdline
    ):
        return "cloudflared"

    auth_script = str(WORKDIR / "scripts" / "auth_proxy.py")
    if (
        auth_script in cmdline
        and "--listen-host 127.0.0.1" in cmdline
        and "--listen-port 8090" in cmdline
        and "--backend-host 127.0.0.1" in cmdline
        and "--backend-port 8088" in cmdline
    ):
        return "auth_proxy"

    if (
        "llama-server" in cmdline
        and "Muse-Glimmer-30B-Q4_K_M.gguf" in cmdline
        and "-a muse-glimmer-30B" in cmdline
        and "--host 127.0.0.1" in cmdline
        and "--port 8088" in cmdline
    ):
        return "llama_server"

    return None


def list_muse_demo_processes():
    p = subprocess.run(
        ["ps", "-eo", "pid=,args="],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    found = []
    if p.returncode != 0:
        return found
    for raw in p.stdout.splitlines():
        raw = raw.strip()
        if not raw:
            continue
        parts = raw.split(None, 1)
        if len(parts) != 2 or not parts[0].isdigit():
            continue
        pid = int(parts[0])
        if pid == os.getpid():
            continue
        kind = classify_muse_demo_process(parts[1])
        if kind:
            found.append((pid, kind))
    return found


def pid_is_alive(pid):
    try:
        stat_path = Path(f"/proc/{pid}/stat")
        if stat_path.is_file():
            fields = stat_path.read_text(encoding="utf-8", errors="replace").split()
            if len(fields) >= 3 and fields[2] == "Z":
                return False
        os.kill(pid, 0)
        return True
    except (ProcessLookupError, FileNotFoundError):
        return False
    except PermissionError:
        return True


def terminate_muse_demo_processes(processes, timeout_seconds=5.0):
    """Terminate only already-classified exact Muse demo PIDs."""
    unique = []
    seen = set()
    for pid, kind in processes:
        if pid in seen:
            continue
        seen.add(pid)
        unique.append((pid, kind))

    for pid, kind in unique:
        try:
            os.kill(pid, signal.SIGTERM)
            print(f"STALE_PROCESS_SIGTERM pid={pid} kind={kind}")
        except ProcessLookupError:
            pass

    deadline = time.monotonic() + float(timeout_seconds)
    while time.monotonic() < deadline:
        if not any(pid_is_alive(pid) for pid, _ in unique):
            break
        time.sleep(0.1)

    killed = []
    for pid, kind in unique:
        if not pid_is_alive(pid):
            continue
        try:
            os.kill(pid, signal.SIGKILL)
            killed.append((pid, kind))
            print(f"STALE_PROCESS_SIGKILL pid={pid} kind={kind}")
        except ProcessLookupError:
            pass

    return {
        "matched": len(unique),
        "sigkill_count": len(killed),
    }


def loopback_port_is_free(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(0.25)
    try:
        return s.connect_ex(("127.0.0.1", int(port))) != 0
    finally:
        s.close()



def install_auth_proxy_timeout_shim():
    """
    Create a notebook-owned python3 launcher outside WORKDIR. It forwards all
    Python calls unchanged except the exact frozen auth_proxy.py launch.
    """
    import shlex

    real_python3 = shutil.which("python3")
    if not real_python3:
        raise RuntimeError("python3 is unavailable for auth-proxy shim.")

    shim_dir = PYTHON_SHIM_DIR
    shim_dir.mkdir(parents=True, exist_ok=True)
    shim_path = shim_dir / "python3"
    auth_proxy_path = WORKDIR / "scripts" / "auth_proxy.py"

    script = f"""#!/usr/bin/env bash
set -Eeuo pipefail
REAL_PYTHON3={shlex.quote(real_python3)}
AUTH_PROXY={shlex.quote(str(auth_proxy_path))}
if [[ "${{1:-}}" == "$AUTH_PROXY" ]]; then
  exec "$REAL_PYTHON3" "$@" --backend-timeout "${{MUSE_AUTH_PROXY_BACKEND_TIMEOUT_SECONDS:?missing timeout override}}"
fi
exec "$REAL_PYTHON3" "$@"
"""
    tmp = shim_dir / f".python3.tmp.{os.getpid()}.{secrets.token_hex(6)}"
    tmp.write_text(script, encoding="utf-8")
    os.chmod(tmp, 0o755)
    os.replace(tmp, shim_path)
    os.chmod(shim_path, 0o755)

    print(f"AUTH_PROXY_TIMEOUT_SHIM={shim_path}")
    print(f"AUTH_PROXY_TIMEOUT_SECONDS={AUTH_PROXY_BACKEND_TIMEOUT_SECONDS}")
    return shim_path, real_python3


def verify_frozen_gateway_timeout_root_cause():
    auth_proxy = (WORKDIR / "scripts" / "auth_proxy.py").read_text(
        encoding="utf-8", errors="replace"
    )
    policy = (WORKDIR / "scripts" / "demo_policy.py").read_text(
        encoding="utf-8", errors="replace"
    )
    expose = (WORKDIR / "expose.sh").read_text(
        encoding="utf-8", errors="replace"
    )

    assert 'default=P.CONNECT_TIMEOUT' in auth_proxy, (
        "Frozen auth_proxy backend-timeout default no longer maps to CONNECT_TIMEOUT."
    )
    assert 'CONNECT_TIMEOUT = 5.0' in policy, (
        "Frozen demo_policy CONNECT_TIMEOUT drifted from the observed 5-second contract."
    )
    assert '--backend-timeout' not in expose, (
        "Frozen expose.sh now supplies an explicit backend timeout; notebook overlay is unnecessary."
    )

    print("FROZEN_GATEWAY_TIMEOUT_ROOT_CAUSE=PASS")
    return {
        "frozen_connect_timeout_seconds": 5.0,
        "expose_supplies_backend_timeout": False,
    }


def verify_auth_proxy_timeout_process(timeout_seconds):
    matches = [
        pid for pid, kind in list_muse_demo_processes()
        if kind == "auth_proxy"
    ]
    assert len(matches) == 1, (
        f"Expected exactly one Muse auth_proxy process, found {matches}"
    )

    pid = matches[0]
    cmdline_path = Path(f"/proc/{pid}/cmdline")
    raw = cmdline_path.read_bytes()
    argv = [
        part.decode("utf-8", errors="replace")
        for part in raw.split(b"\0")
        if part
    ]

    auth_proxy_path = str(WORKDIR / "scripts" / "auth_proxy.py")
    assert auth_proxy_path in argv, (
        "Running auth proxy does not reference the exact frozen auth_proxy.py path."
    )
    assert "--backend-timeout" in argv, (
        "Running auth proxy is missing the v7 backend-timeout override."
    )
    idx = argv.index("--backend-timeout")
    assert idx + 1 < len(argv), "Missing value after --backend-timeout."
    actual = float(argv[idx + 1])
    assert actual == float(timeout_seconds), (
        f"Auth proxy timeout override mismatch: {actual} != {timeout_seconds}"
    )

    print("AUTH_PROXY_TIMEOUT_OVERRIDE_GATE=PASS")
    print(f"AUTH_PROXY_PID={pid}")
    print(f"AUTH_PROXY_BACKEND_TIMEOUT_SECONDS={actual:g}")
    return pid

def repo_config_keys():
    # Parse the pinned common.sh _CONFIG_KEYS block instead of hardcoding it.
    common = WORKDIR / "scripts" / "common.sh"
    text = common.read_text(encoding="utf-8", errors="replace")
    start = text.find("_CONFIG_KEYS=(")
    if start < 0:
        raise RuntimeError("Could not locate _CONFIG_KEYS in scripts/common.sh")
    start += len("_CONFIG_KEYS=(")
    end = text.find("\n)", start)
    if end < 0:
        raise RuntimeError("Could not parse _CONFIG_KEYS block in scripts/common.sh")
    import shlex
    return tuple(shlex.split(text[start:end]))


REPO_DERIVED_ENV_KEYS = {
    "VENV_DIR", "LLAMA_DIR", "MODELS_DIR", "RUNS_DIR", "ARTIFACTS_DIR",
    "COMPARISONS_DIR", "RUNTIME_STATE_DIR",
    "LLAMA_RESOLVED_BIN_DIR", "LLAMA_RUNTIME_SOURCE",
    "LLAMA_RUNTIME_LD_LIBRARY_PATH", "LLAMA_RUNTIME_DATASET_ROOT",
    "LLAMA_RUNTIME_VERSION_TEXT", "LLAMA_SOURCE_SNAPSHOT",
    "LLAMA_SOURCE_PATH", "LLAMA_SOURCE_TREE_SHA256", "LLAMA_BUILD_CMAKE_ARGS",
    "LLAMA_BUILD_COMPILER", "LLAMA_BUILD_CUDA_VERSION", "LLAMA_BUILD_DIR",
    "LLAMA_BINARY_SHA256_SERVER", "LLAMA_BINARY_SHA256_CLI",
    "LLAMA_BINARY_SHA256_BENCH", "LLAMA_RUNTIME_MANIFEST_VERIFIED",
    "LLAMA_RUNTIME_MANIFEST_PATH", "LLAMA_RUNTIME_SOURCE_COMMIT",
    "MODEL_RESOLVED_PATH", "MODEL_RESOLVED_FILE", "MODEL_RUNTIME_SOURCE",
    "MODEL_RUNTIME_DATASET_ROOT", "MODEL_RESOLVED_SIZE_BYTES",
    "MODEL_RESOLVED_SHA256", "MODEL_METADATA_SOURCE", "MODEL_SELECTION",
    "MODEL_VERIFICATION_SECONDS", "SETUP_TOTAL_SECONDS",
    "VENV_BOOTSTRAP_SECONDS", "RUNTIME_DISCOVERY_SECONDS", "VENV_REUSED",
    "LLAMA_SOURCE_BUILD_SECONDS", "SOURCE_BUILD_CACHE_HIT",
    "CUDAToolkit_ROOT", "CUDACXX", "CMAKE_CUDA_COMPILER",
    "CUDA_TOOLKIT_ROOT_RESOLVED", "CUDA_NVCC_PATH", "CUDA_NVCC_REALPATH",
    "CUDA_RUNTIME_HEADER", "CUDA_CUDART_PATH", "CUDA_TOOLKIT_VERSION",
    "CUDA_DRIVER_LIBRARY", "CUDA_PATH",
    "NVIDIA_SMI_PATH", "NVIDIA_SMI_REALPATH", "NVIDIA_SMI_BIN_DIR",
    "NVIDIA_ML_LIBRARY", "NVIDIA_DRIVER_LIB_DIR",
    "NVIDIA_RUNTIME_LD_LIBRARY_PATH", "NVIDIA_RUNTIME_PATH_INJECTED",
    "NVIDIA_RUNTIME_LD_LIBRARY_INJECTED",
    "EXTERNAL_MODE", "TUNNEL_TOKEN", "EXPOSURE_PUBLIC_URL",
    "MUSE_DEMO_TELEMETRY_FILE", "EXTERNAL_LOG_FILE",
}


def build_canonical_demo_env(token_value, cloudflared_path, exposure_mode="proxy"):
    # Preserve Kaggle/system env, remove only repository-owned stale overrides.
    child = os.environ.copy()
    removed = []

    for key in set(repo_config_keys()) | REPO_DERIVED_ENV_KEYS:
        if key in child:
            removed.append(key)
            child.pop(key, None)

    for key in list(child):
        if key.startswith("EXPOSURE_") or key.startswith("EXTERNAL_"):
            removed.append(key)
            child.pop(key, None)

    child["CUDA_VISIBLE_DEVICES"] = "0,1"
    child["SERVE_PROFILE"] = "dflash2"
    if exposure_mode not in {"proxy", "quick", "named"}:
        raise ValueError(f"Unsupported exposure_mode={exposure_mode!r}")
    child["EXPOSURE_MODE"] = exposure_mode
    child["PYTHONUNBUFFERED"] = "1"
    child["MUSE_API_TOKEN"] = token_value
    child["CLOUDFLARED_BIN"] = str(cloudflared_path)
    child["MUSE_AUTH_PROXY_BACKEND_TIMEOUT_SECONDS"] = str(
        AUTH_PROXY_BACKEND_TIMEOUT_SECONDS
    )
    existing_path = child.get("PATH", "")
    child["PATH"] = (
        str(PYTHON_SHIM_DIR)
        + (os.pathsep + existing_path if existing_path else "")
    )

    return child, sorted(set(removed))



def gateway_policy_max_tokens(child_env):
    # Read the cap from the pinned v1.0.0 source itself.
    script = (
        "import sys;"
        "sys.path.insert(0,'scripts');"
        "import demo_policy as P;"
        "print(P.MAX_TOKENS)"
    )
    result = run(
        ["python3", "-B", "-c", script],
        cwd=WORKDIR,
        env=child_env,
        check=True,
    )
    value = int(result.stdout.strip().splitlines()[-1])
    return value

def canonical_effective_config(child_env):
    # Resolve exactly like persistent serve.sh: RUN_PROFILE=balanced.
    keys = [
        "RUN_PROFILE", "CUDA_VISIBLE_DEVICES",
        "GPU_LAYERS", "GPU_SPLIT_MODE", "GPU_TENSOR_SPLIT",
        "PARALLEL_SLOTS", "CONTEXT_SIZE", "BATCH_SIZE", "UBATCH_SIZE",
        "DFLASH_DRAFT_N_MAX", "DFLASH_DRAFT_GPU_LAYERS",
        "MAX_TOKENS", "REASONING_STRENGTH", "REASONING_BUDGET",
        "MIN_ANSWER_TOKENS", "PROMPT_LIMIT",
    ]
    shell = r'''
set -Eeuo pipefail
source scripts/common.sh
RUN_PROFILE=balanced
load_profile_exports
for k in RUN_PROFILE CUDA_VISIBLE_DEVICES GPU_LAYERS GPU_SPLIT_MODE GPU_TENSOR_SPLIT \
         PARALLEL_SLOTS CONTEXT_SIZE BATCH_SIZE UBATCH_SIZE DFLASH_DRAFT_N_MAX \
         DFLASH_DRAFT_GPU_LAYERS MAX_TOKENS REASONING_STRENGTH REASONING_BUDGET \
         MIN_ANSWER_TOKENS PROMPT_LIMIT
do
  printf '%s=%s\n' "$k" "${!k-}"
done
'''
    result = run(
        ["bash", "-lc", shell],
        cwd=WORKDIR,
        env=child_env,
        check=True,
    )
    resolved = parse_env_blob(result.stdout)
    return {k: resolved.get(k) for k in keys}


EXPECTED_CANONICAL_EFFECTIVE_CONFIG = {
    "RUN_PROFILE": "balanced",
    "CUDA_VISIBLE_DEVICES": "0,1",
    "GPU_LAYERS": "999",
    "GPU_SPLIT_MODE": "layer",
    "GPU_TENSOR_SPLIT": "1,1",
    "PARALLEL_SLOTS": "1",
    "CONTEXT_SIZE": "8192",
    "BATCH_SIZE": "512",
    "UBATCH_SIZE": "128",
    "DFLASH_DRAFT_N_MAX": "15",
    "DFLASH_DRAFT_GPU_LAYERS": "999",
    "MAX_TOKENS": "640",
    "REASONING_STRENGTH": "medium",
    "REASONING_BUDGET": "192",
    "MIN_ANSWER_TOKENS": "224",
    "PROMPT_LIMIT": "0",
}


def read_tail(path, max_lines=200):
    path = Path(path)
    if not path.is_file():
        return ""
    return "\n".join(
        path.read_text(encoding="utf-8", errors="replace").splitlines()[-max_lines:]
    ) + "\n"


def safe_copy_text(src, dst, max_lines=None):
    src = Path(src)
    dst = Path(dst)
    if not src.is_file():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if max_lines is None:
        text = src.read_text(encoding="utf-8", errors="replace")
    else:
        text = read_tail(src, max_lines=max_lines)
    dst.write_text(text, encoding="utf-8")
    return True


def capture_backend_forensics(stage, response=None, env=None):
    # Capture bounded operational evidence only; never write bearer tokens.
    SESSION["forensic_capture_count"] = int(SESSION.get("forensic_capture_count", 0)) + 1
    child_env = (env or os.environ.copy()).copy()
    child_env.pop("TUNNEL_TOKEN", None)

    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    stage_dir = FORENSIC_DIR / f"{stamp}-{stage}"
    stage_dir.mkdir(parents=True, exist_ok=True)

    summary = {
        "stage": stage,
        "captured_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "response_http": getattr(response, "status_code", None),
        "response_error_code": None,
        "response_error_type": None,
        "response_request_id": None,
        "port_8088_open": not loopback_port_is_free(8088),
        "port_8090_open": not loopback_port_is_free(8090),
        "muse_processes": [
            {"pid": pid, "kind": kind}
            for pid, kind in list_muse_demo_processes()
        ],
    }

    if response is not None:
        try:
            payload = response.json()
            err = payload.get("error") or {}
            summary["response_error_code"] = err.get("code")
            summary["response_error_type"] = err.get("type")
            summary["response_request_id"] = (
                payload.get("request_id") or err.get("request_id")
            )
        except Exception:
            pass

    commands = {
        "external-status.env": ["./external.sh", "status", "--format", "env"],
        "serve-status.txt": ["./serve.sh", "status"],
        "nvidia-smi.txt": [
            "nvidia-smi",
            "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
            "--format=csv,noheader,nounits",
        ],
    }

    for filename, args in commands.items():
        result = run(
            args,
            cwd=WORKDIR if args[0].startswith("./") else None,
            env=child_env,
            check=False,
        )
        (stage_dir / filename).write_text(
            result.stdout or "",
            encoding="utf-8",
        )

    (stage_dir / "muse-processes.txt").write_text(
        "".join(
            f"pid={item['pid']} kind={item['kind']}\n"
            for item in summary["muse_processes"]
        ),
        encoding="utf-8",
    )

    state_dir = WORKDIR / "artifacts" / "runtime-state"

    for name in (
        "server.json",
        "server-metadata.json",
        "exposure.json",
        "demo-observability.env",
        "demo-observability.json",
        "llama.env",
        "model.env",
        "setup.env",
        "nvidia.env",
    ):
        safe_copy_text(state_dir / name, stage_dir / name)

    for name in (
        "server.log",
        "exposure-proxy.log",
        "exposure-tunnel.log",
        "external.log",
    ):
        safe_copy_text(
            state_dir / name,
            stage_dir / f"{name}.tail.txt",
            max_lines=250,
        )

    (stage_dir / "forensic-summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    (FORENSIC_DIR / "latest-summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )

    if FORENSIC_ZIP_PATH.exists():
        FORENSIC_ZIP_PATH.unlink()
    shutil.make_archive(
        str(FORENSIC_ZIP_PATH.with_suffix("")),
        "zip",
        root_dir=FORENSIC_DIR,
    )

    print(f"FORENSIC_CAPTURE=PASS stage={stage}")
    print(f"FORENSIC_DIR={stage_dir}")
    print(f"FORENSIC_ZIP={FORENSIC_ZIP_PATH}")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary

def parse_env_blob(text):
    out = {}
    for raw in text.splitlines():
        if "=" not in raw:
            continue
        key, value = raw.split("=", 1)
        key = key.strip()
        if key:
            out[key] = value.strip()
    return out

print("CONFIG=READY")
print(f"TAG={TAG}")
print(f"EXPECTED_HEAD={EXPECTED_HEAD}")
print(f"AUTO_CLEANUP_AT_END={AUTO_CLEANUP_AT_END}")
print(f"AUTO_DELETE_GENERATED_TOKEN_AT_END={AUTO_DELETE_GENERATED_TOKEN_AT_END}")
print(f"PUBLIC_DNS_RETRY_SECONDS={PUBLIC_DNS_RETRY_SECONDS}")
print(f"DEMO_REQUEST_MAX_TOKENS={DEMO_REQUEST_MAX_TOKENS}")
print(f"DEMO_REASONING_STRENGTH={DEMO_REASONING_STRENGTH}")
print(f"AUTH_PROXY_BACKEND_TIMEOUT_SECONDS={AUTH_PROXY_BACKEND_TIMEOUT_SECONDS}")


## 2. Verify the Kaggle NVIDIA T4 x2 hardware gate / Xác minh cổng phần cứng Kaggle NVIDIA T4 x2

**English:** This cell verifies that the notebook is running on Kaggle with exactly two NVIDIA T4 GPUs and at least 14,000 MiB visible VRAM per GPU. A successful qualification ends with `KAGGLE_T4X2_GATE=PASS`. If this gate fails, do not continue with the real production-demo run.

**Tiếng Việt:** Cell này xác minh notebook đang chạy trên Kaggle với đúng hai GPU NVIDIA T4 và mỗi GPU có ít nhất 14.000 MiB VRAM khả dụng. Khi đạt yêu cầu, output kết thúc bằng `KAGGLE_T4X2_GATE=PASS`. Nếu gate này FAIL thì không nên tiếp tục real production-demo.


In [ ]:
assert Path("/kaggle/working").is_dir(), "This notebook is intended for Kaggle."
assert INPUT_ROOT.is_dir(), "/kaggle/input is missing."

query = run([
    "nvidia-smi",
    "--query-gpu=name,memory.total",
    "--format=csv,noheader,nounits",
]).stdout.strip().splitlines()

gpus = []
for row in query:
    name, mem = [x.strip() for x in row.split(",", 1)]
    gpus.append({"name": name, "memory_mib": int(mem)})

print(json.dumps(gpus, indent=2))
assert len(gpus) == 2, f"Expected exactly 2 GPUs, found {len(gpus)}"
assert all("T4" in x["name"] for x in gpus), f"Expected NVIDIA T4 x2, found {gpus}"
assert all(x["memory_mib"] >= 14000 for x in gpus), f"Each GPU must expose >=14000 MiB, found {gpus}"

SESSION["gpus"] = gpus
print("KAGGLE_T4X2_GATE=PASS")


## 3. Discover the three required attached Kaggle inputs / Tìm ba Kaggle input bắt buộc

**English:** This cell recursively discovers the exact target GGUF, exact DFlash2 draft GGUF, and prebuilt llama runtime dataset under `/kaggle/input`. Before running it, attach the canonical resources listed in **Before you run**: target `dangkhoa2016/bartowski-muse-glimmer-30b-gguf` → GGUF/`q4-k-m`/1, DFlash2 `dangkhoa2016/incoai-muse-glimmer-30b-dflash2-gguf` → GGUF/`default`/1, and dataset `dangkhoa2016/muse-glimmer-30b-dflash2-llama-runtime`. Recursive discovery is required because Kaggle may mount models and datasets under nested directories. Success is `ATTACHED_INPUT_GATE=PASS`.

**Tiếng Việt:** Cell này tìm đệ quy exact target GGUF, exact DFlash2 draft GGUF và prebuilt llama runtime dataset bên dưới `/kaggle/input`. Trước khi chạy, hãy attach các resource canonical đã ghi trong **Trước khi chạy**: target `dangkhoa2016/bartowski-muse-glimmer-30b-gguf` → GGUF/`q4-k-m`/1, DFlash2 `dangkhoa2016/incoai-muse-glimmer-30b-dflash2-gguf` → GGUF/`default`/1 và dataset `dangkhoa2016/muse-glimmer-30b-dflash2-llama-runtime`. Việc tìm đệ quy là bắt buộc vì Kaggle có thể mount model/dataset trong các thư mục lồng nhau. Thành công được xác nhận bằng `ATTACHED_INPUT_GATE=PASS`.


In [ ]:
TARGET_NAME = "Muse-Glimmer-30B-Q4_K_M.gguf"
DRAFT_NAME = "Muse-Glimmer-30B-DFlash2-Q4_K_M.gguf"
RUNTIME_SLUG = "muse-glimmer-30b-dflash2-llama-runtime"

# Kaggle can mount inputs below nested roots such as:
#   /kaggle/input/models/<owner>/<model>/...
#   /kaggle/input/datasets/<owner>/<dataset>/...
# Discover recursively rather than assuming the slug is a direct child of /kaggle/input.
targets = sorted(p for p in INPUT_ROOT.rglob(TARGET_NAME) if p.is_file())
drafts = sorted(p for p in INPUT_ROOT.rglob(DRAFT_NAME) if p.is_file())
runtime_roots = sorted(
    p for p in INPUT_ROOT.rglob(RUNTIME_SLUG)
    if p.is_dir()
)

top_inputs = sorted(p.name for p in INPUT_ROOT.iterdir() if p.is_dir())

print("Attached top-level Kaggle input roots:")
for name in top_inputs:
    print(" -", name)

print("\nTarget matches:")
for p in targets:
    print(" -", p)

print("\nDFlash2 draft matches:")
for p in drafts:
    print(" -", p)

print("\nPrebuilt runtime dataset roots:")
for p in runtime_roots:
    print(" -", p)

assert targets, f"Missing exact target filename anywhere below {INPUT_ROOT}: {TARGET_NAME}"
assert drafts, f"Missing exact draft filename anywhere below {INPUT_ROOT}: {DRAFT_NAME}"
assert runtime_roots, (
    f"Missing runtime dataset directory named '{RUNTIME_SLUG}' anywhere below {INPUT_ROOT}. "
    "Attach the canonical runtime dataset for the production-demo path."
)

SESSION["target_match_count"] = len(targets)
SESSION["draft_match_count"] = len(drafts)
SESSION["prebuilt_runtime_root_count"] = len(runtime_roots)
SESSION["runtime_roots"] = [str(p) for p in runtime_roots]

print("ATTACHED_INPUT_GATE=PASS")


## 4. Recover safely, clear stale failure residue, and verify the frozen source / Khôi phục an toàn, dọn residue lỗi cũ và xác minh frozen source

**English:** Before deleting any previous worktree, this cell attempts canonical `stop-all`, scans only exact Muse demo process signatures, terminates only matched owned processes, and refuses to kill unknown listeners on ports 8088/8090. After the process/port gates are clean, it removes forensic artifacts left by an earlier run so evidence from the new qualification cannot be confused with stale failure data. It then fresh-clones tag `v1.0.0`, verifies the exact HEAD/tree, checks a clean worktree, and validates `SOURCE_MANIFEST.sha256`. Key PASS lines include `RERUN_RECOVERY=PASS`, `STALE_FORENSIC_CLEANUP=PASS`, and `FROZEN_SOURCE_IDENTITY=PASS`.

**Tiếng Việt:** Trước khi xóa worktree cũ, cell này thử canonical `stop-all`, chỉ quét các process signature chính xác của Muse demo, chỉ dừng process đã nhận diện là thuộc demo và từ chối kill listener lạ trên port 8088/8090. Sau khi process/port gate sạch, cell xóa forensic artifact còn sót từ run trước để evidence của lần qualification mới không bị nhầm với dữ liệu lỗi cũ. Sau đó cell clone mới tag `v1.0.0`, xác minh exact HEAD/tree, worktree sạch và kiểm tra `SOURCE_MANIFEST.sha256`. Các dòng PASS quan trọng gồm `RERUN_RECOVERY=PASS`, `STALE_FORENSIC_CLEANUP=PASS` và `FROZEN_SOURCE_IDENTITY=PASS`.


In [ ]:
# Safe rerun recovery MUST happen before deleting the old worktree/state.
canonical_stop_attempted = False
canonical_stop_rc = None

previous_external = WORKDIR / "external.sh"
if previous_external.is_file() and os.access(previous_external, os.X_OK):
    canonical_stop_attempted = True
    print("RERUN_RECOVERY_CANONICAL_STOP=ATTEMPT")
    stop_result = run(
        ["./external.sh", "stop-all", "--format", "env"],
        cwd=WORKDIR,
        env=os.environ.copy(),
        check=False,
    )
    canonical_stop_rc = stop_result.returncode
    print(f"RERUN_RECOVERY_CANONICAL_STOP_RC={canonical_stop_rc}")

# Give a canonical stop a short moment to reap owned children before orphan scan.
time.sleep(0.5)

stale_before = list_muse_demo_processes()
print(f"RERUN_RECOVERY_MATCHED_ORPHANS={len(stale_before)}")
for pid, kind in stale_before:
    print(f"RERUN_RECOVERY_ORPHAN pid={pid} kind={kind}")

termination = {
    "matched": 0,
    "sigkill_count": 0,
}
if stale_before:
    termination = terminate_muse_demo_processes(stale_before, timeout_seconds=5.0)

time.sleep(0.25)
stale_after = list_muse_demo_processes()
assert not stale_after, (
    "Exact Muse demo processes are still alive after bounded recovery: "
    f"{stale_after}"
)

port_8088_free = loopback_port_is_free(8088)
port_8090_free = loopback_port_is_free(8090)

print(f"PORT_8088_FREE_BEFORE_START={'PASS' if port_8088_free else 'FAIL'}")
print(f"PORT_8090_FREE_BEFORE_START={'PASS' if port_8090_free else 'FAIL'}")

assert port_8088_free, (
    "127.0.0.1:8088 is still occupied after exact Muse recovery. "
    "Refusing to kill an unknown/foreign listener."
)
assert port_8090_free, (
    "127.0.0.1:8090 is still occupied after exact Muse recovery. "
    "Refusing to kill an unknown/foreign listener."
)

# Publication hygiene: after exact owned-process recovery and port gates are clean,
# remove forensic artifacts from an earlier run. Any new failure in this run will
# recreate them and increment SESSION["forensic_capture_count"].
stale_forensic_dir_removed = False
stale_forensic_zip_removed = False

if FORENSIC_DIR.exists():
    shutil.rmtree(FORENSIC_DIR)
    stale_forensic_dir_removed = True

if FORENSIC_ZIP_PATH.exists():
    FORENSIC_ZIP_PATH.unlink()
    stale_forensic_zip_removed = True

SESSION["stale_forensic_dir_removed_before_run"] = stale_forensic_dir_removed
SESSION["stale_forensic_zip_removed_before_run"] = stale_forensic_zip_removed
SESSION["stale_forensic_cleanup"] = "PASS"

print(f"STALE_FORENSIC_DIR_REMOVED={stale_forensic_dir_removed}")
print(f"STALE_FORENSIC_ZIP_REMOVED={stale_forensic_zip_removed}")
print("STALE_FORENSIC_CLEANUP=PASS")

SESSION["rerun_recovery_canonical_stop_attempted"] = canonical_stop_attempted
SESSION["rerun_recovery_canonical_stop_rc"] = canonical_stop_rc
SESSION["rerun_recovery_orphan_match_count"] = len(stale_before)
SESSION["rerun_recovery_sigkill_count"] = termination["sigkill_count"]
SESSION["port_8088_free_before_start"] = port_8088_free
SESSION["port_8090_free_before_start"] = port_8090_free
SESSION["rerun_recovery"] = "PASS"

print("RERUN_RECOVERY=PASS")

# Only now is it safe to discard the previous source/state directory.
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

run([
    "git", "clone",
    "--depth", "1",
    "--branch", TAG,
    REPO_URL,
    str(WORKDIR),
])

head = run(["git", "rev-parse", "HEAD"], cwd=WORKDIR).stdout.strip()
tree = run(["git", "rev-parse", "HEAD^{tree}"], cwd=WORKDIR).stdout.strip()
status = run(["git", "status", "--porcelain"], cwd=WORKDIR).stdout.strip()

print(f"HEAD={head}")
print(f"TREE={tree}")
print(f"WORKTREE_CLEAN={status == ''}")

assert head == EXPECTED_HEAD, f"Tag/HEAD drift: {head} != {EXPECTED_HEAD}"
assert tree == EXPECTED_TREE, f"Tree drift: {tree} != {EXPECTED_TREE}"
assert status == "", "Fresh clone is unexpectedly dirty."

manifest = run(
    ["python3", "-B", "scripts/source_manifest.py", "check", "--root", "."],
    cwd=WORKDIR,
).stdout
assert "SOURCE_MANIFEST: OK" in manifest

SESSION["actual_head"] = head
SESSION["actual_tree"] = tree
SESSION["source_manifest"] = "PASS"
print("FROZEN_SOURCE_IDENTITY=PASS")


## 5. Load or securely generate `MUSE_API_TOKEN` / Nạp hoặc tạo an toàn `MUSE_API_TOKEN`

**English:** This cell first reads the Kaggle Secret `MUSE_API_TOKEN`. If it is missing or invalid, it generates a cryptographically secure temporary token, stores it under `/kaggle/working/.muse-secrets/MUSE_API_TOKEN` with mode `0600`, and never prints the token value. Success is `MUSE_API_TOKEN_VALIDATION=PASS`. Do not publish the generated secret file in notebook outputs.

**Tiếng Việt:** Cell này ưu tiên đọc Kaggle Secret `MUSE_API_TOKEN`. Nếu secret thiếu hoặc không hợp lệ, notebook tự tạo token tạm thời bằng cơ chế mật mã an toàn, lưu tại `/kaggle/working/.muse-secrets/MUSE_API_TOKEN` với mode `0600` và không bao giờ in giá trị token. Thành công là `MUSE_API_TOKEN_VALIDATION=PASS`. Không được publish file secret được tạo tự động trong Notebook Outputs.


In [ ]:
token = None
secret_source = None
GENERATED_TOKEN_CREATED_THIS_RUN = False

def valid_muse_secret(value):
    return (
        isinstance(value, str)
        and len(value) >= 32
        and all(ord(c) >= 32 and ord(c) != 127 for c in value)
    )

# Prefer the operator-managed Kaggle Secret.
try:
    from kaggle_secrets import UserSecretsClient
    candidate = UserSecretsClient().get_secret("MUSE_API_TOKEN")
    if valid_muse_secret(candidate):
        token = candidate
        secret_source = "kaggle_secret"
except Exception:
    token = None

# Safe fallback: generate a temporary secret without ever printing its value.
if token is None:
    GENERATED_SECRET_DIR.mkdir(parents=True, exist_ok=True)
    os.chmod(GENERATED_SECRET_DIR, 0o700)

    token = secrets.token_urlsafe(48)
    assert valid_muse_secret(token), "Generated MUSE_API_TOKEN failed validation."

    fd = os.open(
        str(GENERATED_TOKEN_FILE),
        os.O_WRONLY | os.O_CREAT | os.O_TRUNC,
        0o600,
    )
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            f.write(token + "\n")
    except Exception:
        try:
            os.close(fd)
        except OSError:
            pass
        raise

    os.chmod(GENERATED_TOKEN_FILE, 0o600)
    file_mode = GENERATED_TOKEN_FILE.stat().st_mode & 0o777
    assert file_mode == 0o600, f"Generated token file mode is {oct(file_mode)}, expected 0o600."

    secret_source = "auto_generated_file"
    GENERATED_TOKEN_CREATED_THIS_RUN = True

    print("WARNING: Kaggle Secret MUSE_API_TOKEN was not configured.")
    print("A temporary secure token was generated automatically.")
    print(f"TOKEN_FILE={GENERATED_TOKEN_FILE}")
    print("TOKEN_FILE_MODE=0600")
    print("IMPORTANT: The token value is NOT printed.")
    print("IMPORTANT: Do not publish or include this secret file in Notebook Outputs.")
    print(
        "IMPORTANT: Normal cleanup deletes this file when "
        "AUTO_DELETE_GENERATED_TOKEN_AT_END=True."
    )

assert valid_muse_secret(token)

os.environ["MUSE_API_TOKEN"] = token

SESSION["secret_source"] = secret_source
SESSION["secret_value_persisted"] = (secret_source == "auto_generated_file")
SESSION["secret_file_path"] = (
    str(GENERATED_TOKEN_FILE) if secret_source == "auto_generated_file" else None
)
SESSION["secret_file_mode"] = (
    "0600" if secret_source == "auto_generated_file" else None
)
SESSION["generated_token_cleanup_enabled"] = bool(
    secret_source == "auto_generated_file"
    and AUTO_DELETE_GENERATED_TOKEN_AT_END
)

print(f"MUSE_API_TOKEN_SOURCE={secret_source}")
print("MUSE_API_TOKEN_VALIDATION=PASS")
print("MUSE_API_TOKEN_VALUE=REDACTED")


## 6. Prepare a safe `cloudflared` binary / Chuẩn bị binary `cloudflared` an toàn

**English:** This cell uses a valid system `cloudflared` when available, otherwise reuses a valid cached binary, or downloads to a temporary file and atomically replaces the canonical cache path. This prevents overwriting an executing binary during reruns. Success is `CLOUDFLARED_GATE=PASS`; the printed SHA-256 is for the binary only and contains no secret.

**Tiếng Việt:** Cell này dùng `cloudflared` của hệ thống nếu hợp lệ; nếu không thì tái sử dụng binary cache hợp lệ hoặc download vào file tạm rồi thay thế atomically đường dẫn cache canonical. Cách này tránh ghi đè một executable đang chạy khi rerun. Thành công là `CLOUDFLARED_GATE=PASS`; SHA-256 được in chỉ thuộc binary và không chứa secret.


In [ ]:
BIN_DIR.mkdir(parents=True, exist_ok=True)

download_url = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64"
)

system_cloudflared = shutil.which("cloudflared")
canonical_cloudflared = BIN_DIR / "cloudflared"
cloudflared = None
CLOUDFLARED_BOOTSTRAP_MODE = None

if system_cloudflared:
    probe = run([system_cloudflared, "--version"], check=False)
    if probe.returncode == 0:
        cloudflared = system_cloudflared
        CLOUDFLARED_BOOTSTRAP_MODE = "system"

if cloudflared is None and canonical_cloudflared.is_file():
    try:
        os.chmod(canonical_cloudflared, 0o755)
    except OSError:
        pass
    probe = run([str(canonical_cloudflared), "--version"], check=False)
    if probe.returncode == 0:
        cloudflared = str(canonical_cloudflared)
        CLOUDFLARED_BOOTSTRAP_MODE = "reuse_cached"

if cloudflared is None:
    temp_cloudflared = BIN_DIR / (
        f".cloudflared.tmp.{os.getpid()}.{secrets.token_hex(6)}"
    )
    try:
        if temp_cloudflared.exists():
            temp_cloudflared.unlink()

        run([
            "curl", "-fL", "--retry", "3", "--connect-timeout", "20",
            download_url, "-o", str(temp_cloudflared)
        ])

        os.chmod(temp_cloudflared, 0o755)

        temp_probe = run([str(temp_cloudflared), "--version"], check=False)
        assert temp_probe.returncode == 0, (
            "Downloaded cloudflared temporary binary failed --version."
        )

        # Atomic path replacement: never open the canonical executable path for writing.
        os.replace(temp_cloudflared, canonical_cloudflared)
        os.chmod(canonical_cloudflared, 0o755)

        cloudflared = str(canonical_cloudflared)
        CLOUDFLARED_BOOTSTRAP_MODE = "atomic_download"
    finally:
        if temp_cloudflared.exists():
            temp_cloudflared.unlink()

assert cloudflared is not None
assert Path(cloudflared).is_file(), "cloudflared binary is unavailable."

os.environ["CLOUDFLARED_BIN"] = cloudflared

version = run([cloudflared, "--version"]).stdout.strip()
sha = hashlib.sha256(Path(cloudflared).read_bytes()).hexdigest()

print(f"CLOUDFLARED_BOOTSTRAP_MODE={CLOUDFLARED_BOOTSTRAP_MODE}")
print(f"CLOUDFLARED_BIN={cloudflared}")
print(f"CLOUDFLARED_VERSION={version}")
print(f"CLOUDFLARED_SHA256={sha}")

SESSION["cloudflared_bootstrap_mode"] = CLOUDFLARED_BOOTSTRAP_MODE
SESSION["cloudflared_version"] = version
SESSION["cloudflared_sha256"] = sha

print("CLOUDFLARED_GATE=PASS")


## 7. Build the canonical environment and run readiness/preflight / Dựng môi trường canonical và chạy readiness/preflight

**English:** This cell proves the frozen 5-second gateway timeout root cause, installs the notebook-owned `python3` shim that adds `--backend-timeout 120` only to the exact frozen `auth_proxy.py`, removes stale repository-owned environment overrides, verifies the canonical `balanced` serving profile (`CONTEXT_SIZE=8192`), enforces the 512-token public cap, and runs repository readiness/preflight in local `proxy` mode. Expected gates include `DEMO_REQUEST_BUDGET_GATE=PASS`, `AUTH_PROXY_TIMEOUT_OVERLAY=PASS`, `CANONICAL_ENVIRONMENT=PASS`, `READINESS=PASS`, and `EXTERNAL_PREFLIGHT=PASS`.

**Tiếng Việt:** Cell này chứng minh root cause timeout gateway 5 giây của frozen source, cài `python3` shim do notebook quản lý để chỉ thêm `--backend-timeout 120` cho đúng frozen `auth_proxy.py`, loại bỏ các environment override cũ thuộc repository, xác minh serving profile canonical `balanced` (`CONTEXT_SIZE=8192`), giữ public cap 512 token và chạy readiness/preflight ở chế độ local `proxy`. Các gate mong đợi gồm `DEMO_REQUEST_BUDGET_GATE=PASS`, `AUTH_PROXY_TIMEOUT_OVERLAY=PASS`, `CANONICAL_ENVIRONMENT=PASS`, `READINESS=PASS` và `EXTERNAL_PREFLIGHT=PASS`.


In [ ]:
timeout_root_cause = verify_frozen_gateway_timeout_root_cause()
python_shim_path, real_python3 = install_auth_proxy_timeout_shim()

env, removed_repo_overrides = build_canonical_demo_env(
    token, cloudflared, exposure_mode="proxy"
)

assert env.get("MUSE_API_TOKEN") == token
assert env.get("CLOUDFLARED_BIN") == cloudflared

print(f"CANONICAL_ENV_REMOVED_OVERRIDE_COUNT={len(removed_repo_overrides)}")
if removed_repo_overrides:
    print("CANONICAL_ENV_REMOVED_KEYS=" + ",".join(removed_repo_overrides))

effective_config = canonical_effective_config(env)

print("CANONICAL_EFFECTIVE_CONFIG_BEGIN")
for key in sorted(effective_config):
    print(f"{key}={effective_config[key]}")
print("CANONICAL_EFFECTIVE_CONFIG_END")

assert effective_config == EXPECTED_CANONICAL_EFFECTIVE_CONFIG, (
    "Persistent serving config does not resolve to the exact pinned v1.0.0 "
    f"balanced profile.\nexpected={EXPECTED_CANONICAL_EFFECTIVE_CONFIG}\n"
    f"actual={effective_config}"
)


gateway_max_tokens = gateway_policy_max_tokens(env)
minimum_visible_answer_reserve = int(effective_config["MIN_ANSWER_TOKENS"])

print(f"GATEWAY_POLICY_MAX_TOKENS={gateway_max_tokens}")
print(f"DEMO_MIN_VISIBLE_ANSWER_RESERVE={minimum_visible_answer_reserve}")
print(f"DEMO_REQUEST_MAX_TOKENS={DEMO_REQUEST_MAX_TOKENS}")
print(f"DEMO_REASONING_STRENGTH={DEMO_REASONING_STRENGTH}")

assert gateway_max_tokens == 512, (
    f"Pinned public gateway max_tokens cap drifted: {gateway_max_tokens} != 512"
)
assert DEMO_REQUEST_MAX_TOKENS <= gateway_max_tokens, (
    "Demo request exceeds the pinned public gateway max_tokens cap."
)
assert DEMO_REQUEST_MAX_TOKENS >= minimum_visible_answer_reserve, (
    "Demo request token budget is smaller than the canonical visible-answer reserve."
)
assert DEMO_REASONING_STRENGTH == "low"

SESSION["gateway_policy_max_tokens"] = gateway_max_tokens
SESSION["demo_min_visible_answer_reserve"] = minimum_visible_answer_reserve
SESSION["demo_request_max_tokens"] = DEMO_REQUEST_MAX_TOKENS
SESSION["demo_reasoning_strength"] = DEMO_REASONING_STRENGTH
SESSION["demo_reasoning_control"] = "chat_template_kwargs.reasoning_strength"
SESSION["demo_request_budget_gate"] = "PASS"
SESSION["frozen_gateway_connect_timeout_seconds"] = timeout_root_cause[
    "frozen_connect_timeout_seconds"
]
SESSION["auth_proxy_backend_timeout_override_seconds"] = (
    AUTH_PROXY_BACKEND_TIMEOUT_SECONDS
)
SESSION["auth_proxy_timeout_shim_path"] = str(python_shim_path)
SESSION["auth_proxy_timeout_overlay"] = "PASS"
SESSION["core_exposure_mode"] = "proxy"

print("DEMO_REQUEST_BUDGET_GATE=PASS")
print("REQUEST_REASONING_POLICY=LOW")
print("AUTH_PROXY_TIMEOUT_OVERLAY=PASS")
print("CORE_EXPOSURE_MODE=proxy")

SESSION["canonical_env_removed_override_count"] = len(removed_repo_overrides)
SESSION["canonical_env_removed_keys"] = removed_repo_overrides
SESSION["canonical_effective_config"] = effective_config
SESSION["canonical_environment"] = "PASS"

print("CANONICAL_ENVIRONMENT=PASS")

readiness = run(
    ["./readiness.sh", "--format", "env"],
    cwd=WORKDIR,
    env=env,
).stdout

preflight = run(
    ["./external.sh", "preflight"],
    cwd=WORKDIR,
    env=env,
).stdout

SESSION["readiness_exit"] = 0
SESSION["external_preflight_exit"] = 0
print("READINESS=PASS")
print("EXTERNAL_PREFLIGHT=PASS")


## Start the production-style demo / Khởi động demo theo phong cách production

### English

The next cell starts **the local production core only**:

- `llama-server` loopback backend on `127.0.0.1:8088`;
- DFlash2 speculative decoding;
- Bearer-authenticated gateway on `127.0.0.1:8090`;
- **no Cloudflare Quick Tunnel dependency in this phase**.

After the local auth smoke, real non-streaming generation, and SSE generation have been qualified, a later dedicated phase attempts the optional Cloudflare Quick Tunnel transport separately.

The repository itself performs the strict source/runtime/model identity checks before serving.

If any later cell fails after startup, run the final **Cleanup** cell manually.

---

### Tiếng Việt

Cell tiếp theo khởi động **chỉ local production core**:

- `llama-server` loopback backend tại `127.0.0.1:8088`;
- speculative decoding bằng DFlash2;
- Bearer-authenticated gateway tại `127.0.0.1:8090`;
- **phase này không phụ thuộc vào Cloudflare Quick Tunnel**.

Sau khi local auth smoke, real non-streaming generation và SSE generation được qualification thành công, một phase riêng ở phía sau mới thử transport qua Cloudflare Quick Tunnel tùy chọn.

Chính repository thực hiện các kiểm tra nghiêm ngặt về source/runtime/model identity trước khi phục vụ request.

Nếu bất kỳ cell nào phía sau bị lỗi sau khi startup, hãy chạy thủ công cell **Cleanup** cuối cùng.


## 8. Start the authenticated local production core / Khởi động local production core có xác thực

**English:** This cell starts only the backend plus authenticated loopback gateway with `EXPOSURE_MODE=proxy`. It then verifies the live auth-proxy process really carries the 120-second timeout override and re-checks the frozen source manifest after startup. No Quick Tunnel is required here. Success includes `AUTH_PROXY_TIMEOUT_OVERRIDE_GATE=PASS`, `POST_START_SOURCE_MANIFEST=PASS`, `CORE_PROXY_START=PASS`, and local endpoint `http://127.0.0.1:8090`.

**Tiếng Việt:** Cell này chỉ khởi động backend và authenticated loopback gateway với `EXPOSURE_MODE=proxy`. Sau đó cell xác minh process auth proxy đang chạy thực sự có timeout override 120 giây và kiểm tra lại frozen source manifest sau startup. Phase này không cần Quick Tunnel. Thành công gồm `AUTH_PROXY_TIMEOUT_OVERRIDE_GATE=PASS`, `POST_START_SOURCE_MANIFEST=PASS`, `CORE_PROXY_START=PASS` và local endpoint `http://127.0.0.1:8090`.


In [ ]:
# Core phase: start only the authenticated local proxy. Public transport is
# qualified later, after real model generation/SSE.
env, _removed_at_start = build_canonical_demo_env(
    token,
    cloudflared,
    exposure_mode="proxy",
)

assert "token" in globals(), (
    "Secret setup cell has not run. Run the MUSE_API_TOKEN cell before starting."
)
assert env.get("MUSE_API_TOKEN") == token
assert env.get("CLOUDFLARED_BIN")

start_result = run(
    ["./external.sh", "start"],
    cwd=WORKDIR,
    env=env,
).stdout

auth_proxy_pid = verify_auth_proxy_timeout_process(
    AUTH_PROXY_BACKEND_TIMEOUT_SECONDS
)
SESSION["auth_proxy_pid_verified"] = True
SESSION["auth_proxy_timeout_process_gate"] = "PASS"

post_start_manifest = run(
    ["python3", "-B", "scripts/source_manifest.py", "check", "--root", "."],
    cwd=WORKDIR,
    env=env,
).stdout
assert "SOURCE_MANIFEST: OK" in post_start_manifest
SESSION["post_start_source_manifest"] = "PASS"
print("POST_START_SOURCE_MANIFEST=PASS")

status_env = run(
    ["./external.sh", "status", "--format", "env"],
    cwd=WORKDIR,
    env=env,
).stdout
status_map = parse_env_blob(status_env)

assert status_map.get("EXPOSURE_STATUS") in {"proxy-ready", "ready"}, status_map
assert status_map.get("MODE") == "proxy", status_map
assert status_map.get("PROXY_OWNED") == "1", status_map
assert status_map.get("TUNNEL_OWNED") == "0", status_map

LOCAL_URL = run(
    ["./external.sh", "local-endpoint"],
    cwd=WORKDIR,
    env=env,
).stdout.strip().rstrip("/")

assert LOCAL_URL == "http://127.0.0.1:8090", LOCAL_URL

PUBLIC_URL = None

SESSION["start_exit"] = 0
SESSION["status_env_captured"] = True
SESSION["local_gateway_url"] = LOCAL_URL
SESSION["core_exposure_mode"] = "proxy"
SESSION["core_proxy_status"] = status_map.get("EXPOSURE_STATUS")

print(f"LOCAL_GATEWAY={LOCAL_URL}")
print("CORE_PROXY_START=PASS")
print("PRODUCTION_DEMO_START=PASS")


## 9. Verify the authenticated local API boundary / Xác minh biên API local có xác thực

**English:** This cell checks that unauthenticated `/v1/models` is rejected with HTTP 401, while authenticated `/ready` and `/v1/models` return HTTP 200 through `127.0.0.1:8090`. This isolates auth/API correctness from public DNS and tunnel behavior. Success is `LOCAL_AUTH_GATEWAY=PASS`.

**Tiếng Việt:** Cell này kiểm tra `/v1/models` không có Bearer token phải bị từ chối với HTTP 401, trong khi `/ready` và `/v1/models` có xác thực phải trả HTTP 200 qua `127.0.0.1:8090`. Việc này tách correctness của auth/API khỏi DNS và tunnel công khai. Thành công là `LOCAL_AUTH_GATEWAY=PASS`.


In [ ]:
# Mandatory core smoke goes through the authenticated loopback gateway.
# This isolates model/runtime/API correctness from external DNS/tunnel transport.
import requests

AUTH = {"Authorization": f"Bearer {token}"}

unauth_models = requests.get(
    f"{LOCAL_URL}/v1/models",
    timeout=30,
)
print("LOCAL_UNAUTH_V1_MODELS_HTTP=", unauth_models.status_code)
assert unauth_models.status_code == 401, (
    f"Expected 401 from local protected /v1/models without Bearer token, "
    f"got {unauth_models.status_code}"
)

ready = requests.get(
    f"{LOCAL_URL}/ready",
    headers=AUTH,
    timeout=60,
)
print("LOCAL_AUTH_READY_HTTP=", ready.status_code)
print("LOCAL_AUTH_READY_BODY=", ready.text.strip()[:500])
assert ready.status_code == 200, ready.text

models = requests.get(
    f"{LOCAL_URL}/v1/models",
    headers=AUTH,
    timeout=60,
)
print("LOCAL_AUTH_V1_MODELS_HTTP=", models.status_code)
assert models.status_code == 200, models.text
models_json = models.json()
print(json.dumps(models_json, indent=2)[:2000])

SESSION["local_unauth_v1_models_http"] = unauth_models.status_code
SESSION["local_auth_ready_http"] = ready.status_code
SESSION["local_auth_v1_models_http"] = models.status_code

print("LOCAL_AUTH_GATEWAY=PASS")


## 10. Run one real non-streaming Muse generation with claim-bounded output / Chạy một real Muse generation non-stream với output được giới hạn claim

**English:** This is the first real production semantic test. It sends `max_tokens=512` together with `chat_template_kwargs.reasoning_strength=low`. The prompt explicitly asks the model to describe only facts directly demonstrated by this notebook and forbids unsupported architecture/security claims. Reasoning content may exist, but visible assistant content must be non-empty and `finish_reason` must be `stop` or `length`. Success is `LOCAL_NONSTREAM_DEMO=PASS`. On HTTP or semantic failure, the notebook captures bounded secret-free forensics and increments the current-run forensic counter.

**Tiếng Việt:** Đây là bài test semantic thực tế đầu tiên của production notebook. Request gửi `max_tokens=512` cùng `chat_template_kwargs.reasoning_strength=low`. Prompt yêu cầu model chỉ mô tả những facts được notebook trực tiếp chứng minh và cấm các claim kiến trúc/bảo mật chưa được xác minh. Reasoning content được phép tồn tại, nhưng visible assistant content phải khác rỗng và `finish_reason` phải là `stop` hoặc `length`. Thành công là `LOCAL_NONSTREAM_DEMO=PASS`. Nếu HTTP hoặc semantic FAIL, notebook thu thập forensic có giới hạn, không chứa secret và tăng bộ đếm forensic của chính run hiện tại.


In [ ]:
LOCAL_NONSTREAM_FINISH_REASON = None
LOCAL_NONSTREAM_STATUS = None
LOCAL_NONSTREAM_FAILURE_CAPTURED = False
LOCAL_NONSTREAM_ERROR_CODE = None
LOCAL_NONSTREAM_ERROR_TYPE = None
LOCAL_NONSTREAM_REQUEST_ID = None
LOCAL_NONSTREAM_SEMANTIC_FAILURE = None
LOCAL_NONSTREAM_CONTENT_CHARS = 0
LOCAL_NONSTREAM_REASONING_CHARS = 0
LOCAL_NONSTREAM_COMPLETION_TOKENS = None
LOCAL_NONSTREAM_PROMPT_TOKENS = None

if RUN_NONSTREAM_DEMO:
    payload = {
        "model": "muse-glimmer-30B",
        "messages": [{
            "role": "user",
            "content": (
                "In exactly two concise sentences, describe only what this notebook "
                "directly verifies: Muse-Glimmer-30B runs on Kaggle NVIDIA T4 x2, "
                "serves through the Bearer-authenticated local gateway, uses the "
                "configured DFlash2 speculative-decoding path, and returns visible "
                "assistant output. Do not claim mTLS, service mesh, HA, rate limiting, "
                "prompt filtering, or any control not explicitly demonstrated here."
            ),
        }],
        "max_tokens": DEMO_REQUEST_MAX_TOKENS,
        "chat_template_kwargs": {"reasoning_strength": DEMO_REASONING_STRENGTH},
    }

    request_started = time.monotonic()
    r = requests.post(
        f"{LOCAL_URL}/v1/chat/completions",
        headers={**AUTH, "Content-Type": "application/json"},
        json=payload,
        timeout=3600,
    )
    request_elapsed = time.monotonic() - request_started

    LOCAL_NONSTREAM_STATUS = r.status_code
    print("LOCAL_NONSTREAM_HTTP=", r.status_code)
    print(f"LOCAL_NONSTREAM_ELAPSED_SECONDS={request_elapsed:.3f}")
    print(f"LOCAL_NONSTREAM_REQUEST_MAX_TOKENS={DEMO_REQUEST_MAX_TOKENS}")
    print(f"LOCAL_NONSTREAM_REASONING_STRENGTH={DEMO_REASONING_STRENGTH}")

    if r.status_code != 200:
        LOCAL_NONSTREAM_FAILURE_CAPTURED = True

        try:
            error_payload = r.json()
            error_obj = error_payload.get("error") or {}
            LOCAL_NONSTREAM_ERROR_CODE = error_obj.get("code")
            LOCAL_NONSTREAM_ERROR_TYPE = error_obj.get("type")
            LOCAL_NONSTREAM_REQUEST_ID = (
                error_payload.get("request_id")
                or error_obj.get("request_id")
            )
        except Exception:
            pass

        print(f"LOCAL_NONSTREAM_ERROR_CODE={LOCAL_NONSTREAM_ERROR_CODE}")
        print(f"LOCAL_NONSTREAM_ERROR_TYPE={LOCAL_NONSTREAM_ERROR_TYPE}")
        print(f"LOCAL_NONSTREAM_REQUEST_ID={LOCAL_NONSTREAM_REQUEST_ID}")

        forensic_summary = capture_backend_forensics(
            "local-nonstream-http-failure",
            response=r,
            env=env,
        )

        SESSION["local_nonstream_forensic_capture"] = "PASS"
        SESSION["local_nonstream_forensic_summary"] = forensic_summary
        print("LOCAL_NONSTREAM_DEMO=FAIL_CAPTURED")
    else:
        data = r.json()
        choices = data.get("choices") or []
        choice = choices[0] if choices else {}
        message = choice.get("message") or {}
        usage = data.get("usage") or {}

        LOCAL_NONSTREAM_FINISH_REASON = choice.get("finish_reason")
        content = message.get("content") or ""

        # Count hidden reasoning structurally without printing/storing its text.
        reasoning_text = (
            message.get("reasoning_content")
            or message.get("reasoning")
            or ""
        )
        if not isinstance(reasoning_text, str):
            reasoning_text = str(reasoning_text)

        LOCAL_NONSTREAM_CONTENT_CHARS = len(content)
        LOCAL_NONSTREAM_REASONING_CHARS = len(reasoning_text)
        LOCAL_NONSTREAM_COMPLETION_TOKENS = usage.get("completion_tokens")
        LOCAL_NONSTREAM_PROMPT_TOKENS = usage.get("prompt_tokens")

        print("\n--- assistant (local gateway) ---")
        print(content)
        print("--- end assistant ---")
        print("finish_reason=", LOCAL_NONSTREAM_FINISH_REASON)
        print("visible_content_chars=", LOCAL_NONSTREAM_CONTENT_CHARS)
        print("reasoning_chars=", LOCAL_NONSTREAM_REASONING_CHARS)
        print("completion_tokens=", LOCAL_NONSTREAM_COMPLETION_TOKENS)

        if not content.strip():
            LOCAL_NONSTREAM_SEMANTIC_FAILURE = "EMPTY_VISIBLE_CONTENT"
        elif LOCAL_NONSTREAM_FINISH_REASON not in {"stop", "length"}:
            LOCAL_NONSTREAM_SEMANTIC_FAILURE = "INVALID_FINISH_REASON"

        if LOCAL_NONSTREAM_SEMANTIC_FAILURE:
            LOCAL_NONSTREAM_FAILURE_CAPTURED = True
            print(
                "LOCAL_NONSTREAM_SEMANTIC_FAILURE="
                + LOCAL_NONSTREAM_SEMANTIC_FAILURE
            )
            forensic_summary = capture_backend_forensics(
                "local-nonstream-semantic-failure",
                response=r,
                env=env,
            )
            SESSION["local_nonstream_forensic_capture"] = "PASS"
            SESSION["local_nonstream_forensic_summary"] = forensic_summary
            print("LOCAL_NONSTREAM_DEMO=FAIL_CAPTURED")
        else:
            print("LOCAL_NONSTREAM_DEMO=PASS")
else:
    print("LOCAL_NONSTREAM_DEMO=SKIPPED")

SESSION["local_nonstream_http"] = LOCAL_NONSTREAM_STATUS
SESSION["local_nonstream_finish_reason"] = LOCAL_NONSTREAM_FINISH_REASON
SESSION["local_nonstream_error_code"] = LOCAL_NONSTREAM_ERROR_CODE
SESSION["local_nonstream_error_type"] = LOCAL_NONSTREAM_ERROR_TYPE
SESSION["local_nonstream_request_id"] = LOCAL_NONSTREAM_REQUEST_ID
SESSION["local_nonstream_failure_captured"] = LOCAL_NONSTREAM_FAILURE_CAPTURED
SESSION["local_nonstream_semantic_failure"] = LOCAL_NONSTREAM_SEMANTIC_FAILURE
SESSION["local_nonstream_content_chars"] = LOCAL_NONSTREAM_CONTENT_CHARS
SESSION["local_nonstream_reasoning_chars"] = LOCAL_NONSTREAM_REASONING_CHARS
SESSION["local_nonstream_completion_tokens"] = LOCAL_NONSTREAM_COMPLETION_TOKENS
SESSION["local_nonstream_prompt_tokens"] = LOCAL_NONSTREAM_PROMPT_TOKENS


SESSION["local_nonstream_reasoning_strength"] = DEMO_REASONING_STRENGTH
SESSION["local_nonstream_visible_answer"] = (
    LOCAL_NONSTREAM_STATUS == 200
    and not LOCAL_NONSTREAM_FAILURE_CAPTURED
    and LOCAL_NONSTREAM_CONTENT_CHARS > 0
)


## 11. Run one real SSE generation with claim-bounded output / Chạy một real SSE generation với output được giới hạn claim

**English:** This cell runs the second real production model qualification through SSE, again using `reasoning_strength=low`. The prompt is intentionally restricted to behavior directly demonstrated by the notebook. Acceptance requires HTTP 200, non-empty visible streamed content, a terminal `finish_reason` of `stop` or `length`, and the `[DONE]` sentinel. If the non-stream test failed, this cell intentionally skips SSE. Success is `LOCAL_SSE_DEMO=PASS`.

**Tiếng Việt:** Cell này chạy qualification thực tế thứ hai của production notebook qua SSE, tiếp tục dùng `reasoning_strength=low`. Prompt được giới hạn rõ ràng vào những hành vi mà notebook trực tiếp chứng minh. Điều kiện đạt gồm HTTP 200, visible streamed content khác rỗng, `finish_reason` cuối là `stop` hoặc `length`, và phải thấy sentinel `[DONE]`. Nếu non-stream test đã FAIL thì cell chủ động bỏ qua SSE. Thành công là `LOCAL_SSE_DEMO=PASS`.


In [ ]:
LOCAL_SSE_STATUS = None
LOCAL_SSE_DONE = False
LOCAL_SSE_FINISH_REASON = None
LOCAL_SSE_CONTENT_CHARS = 0
LOCAL_SSE_REASONING_CHARS = 0
LOCAL_SSE_FAILURE_CAPTURED = False
LOCAL_SSE_SEMANTIC_FAILURE = None

if (
    RUN_SSE_DEMO
    and LOCAL_NONSTREAM_STATUS == 200
    and not LOCAL_NONSTREAM_FAILURE_CAPTURED
):
    payload = {
        "model": "muse-glimmer-30B",
        "messages": [{
            "role": "user",
            "content": (
                "In exactly two concise sentences, describe only what this notebook "
                "directly demonstrates during streaming: DFlash2 is used by the "
                "configured Muse-Glimmer-30B backend, and the Bearer-authenticated "
                "gateway mediates the API path tested by this notebook. Do not claim "
                "mTLS, service mesh, HA, rate limiting, prompt filtering, or any "
                "control not explicitly demonstrated here."
            ),
        }],
        "max_tokens": DEMO_REQUEST_MAX_TOKENS,
        "chat_template_kwargs": {"reasoning_strength": DEMO_REASONING_STRENGTH},
        "stream": True,
    }

    with requests.post(
        f"{LOCAL_URL}/v1/chat/completions",
        headers={**AUTH, "Content-Type": "application/json"},
        json=payload,
        stream=True,
        timeout=3600,
    ) as r:
        LOCAL_SSE_STATUS = r.status_code
        print("LOCAL_SSE_HTTP=", r.status_code)
        print(f"LOCAL_SSE_REQUEST_MAX_TOKENS={DEMO_REQUEST_MAX_TOKENS}")
        print(f"LOCAL_SSE_REASONING_STRENGTH={DEMO_REASONING_STRENGTH}")

        if r.status_code != 200:
            LOCAL_SSE_FAILURE_CAPTURED = True
            capture_backend_forensics(
                "local-sse-http-failure",
                response=r,
                env=env,
            )
            print("LOCAL_SSE_DEMO=FAIL_CAPTURED")
        else:
            print("\n--- assistant stream (local gateway) ---")
            for line in r.iter_lines(decode_unicode=True):
                if not line:
                    continue
                if line == "data: [DONE]":
                    LOCAL_SSE_DONE = True
                    break
                if not line.startswith("data: "):
                    continue

                event = json.loads(line[len("data: "):])
                choices = event.get("choices") or []
                if not choices:
                    continue

                choice = choices[0]
                delta = choice.get("delta") or {}

                piece = delta.get("content") or ""
                if piece:
                    LOCAL_SSE_CONTENT_CHARS += len(piece)
                    print(piece, end="", flush=True)

                reasoning_piece = (
                    delta.get("reasoning_content")
                    or delta.get("reasoning")
                    or ""
                )
                if reasoning_piece:
                    if not isinstance(reasoning_piece, str):
                        reasoning_piece = str(reasoning_piece)
                    LOCAL_SSE_REASONING_CHARS += len(reasoning_piece)

                if choice.get("finish_reason") is not None:
                    LOCAL_SSE_FINISH_REASON = choice["finish_reason"]

            print("\n--- end assistant stream ---")

    if LOCAL_SSE_STATUS == 200:
        print("LOCAL_SSE_DONE=", LOCAL_SSE_DONE)
        print("LOCAL_SSE_FINISH_REASON=", LOCAL_SSE_FINISH_REASON)
        print("LOCAL_SSE_CONTENT_CHARS=", LOCAL_SSE_CONTENT_CHARS)
        print("LOCAL_SSE_REASONING_CHARS=", LOCAL_SSE_REASONING_CHARS)

        if not LOCAL_SSE_DONE:
            LOCAL_SSE_SEMANTIC_FAILURE = "DONE_SENTINEL_NOT_SEEN"
        elif LOCAL_SSE_CONTENT_CHARS <= 0:
            LOCAL_SSE_SEMANTIC_FAILURE = "EMPTY_VISIBLE_CONTENT"
        elif LOCAL_SSE_FINISH_REASON not in {"stop", "length"}:
            LOCAL_SSE_SEMANTIC_FAILURE = "INVALID_FINISH_REASON"

        if LOCAL_SSE_SEMANTIC_FAILURE:
            LOCAL_SSE_FAILURE_CAPTURED = True
            print("LOCAL_SSE_SEMANTIC_FAILURE=" + LOCAL_SSE_SEMANTIC_FAILURE)
            capture_backend_forensics(
                "local-sse-stream-contract-failure",
                env=env,
            )
            print("LOCAL_SSE_DEMO=FAIL_CAPTURED")
        else:
            print("LOCAL_SSE_DEMO=PASS")

elif RUN_SSE_DEMO:
    print("LOCAL_SSE_DEMO=SKIPPED_DUE_TO_NONSTREAM_FAILURE")
else:
    print("LOCAL_SSE_DEMO=SKIPPED")

SESSION["local_sse_http"] = LOCAL_SSE_STATUS
SESSION["local_sse_done"] = LOCAL_SSE_DONE
SESSION["local_sse_finish_reason"] = LOCAL_SSE_FINISH_REASON
SESSION["local_sse_content_chars"] = LOCAL_SSE_CONTENT_CHARS
SESSION["local_sse_reasoning_chars"] = LOCAL_SSE_REASONING_CHARS
SESSION["local_sse_failure_captured"] = LOCAL_SSE_FAILURE_CAPTURED
SESSION["local_sse_semantic_failure"] = LOCAL_SSE_SEMANTIC_FAILURE


SESSION["local_sse_reasoning_strength"] = DEMO_REASONING_STRENGTH
SESSION["local_sse_visible_answer"] = (
    LOCAL_SSE_STATUS == 200
    and not LOCAL_SSE_FAILURE_CAPTURED
    and LOCAL_SSE_CONTENT_CHARS > 0
)


## Public Quick Tunnel transport qualification / Qualification transport qua Public Quick Tunnel

### English

The core production demo has already exercised the real model, DFlash2, authenticated gateway, non-streaming generation, and SSE through loopback.

This step qualifies only the **external transport**. Quick Tunnel DNS can be transient, so the notebook retries DNS + HTTP for up to `PUBLIC_DNS_RETRY_SECONDS` while re-checking:

- external exposure remains `ready`;
- tunnel remains owned/alive;
- the public URL has not changed;
- public unauthenticated `/v1/models` returns `401`;
- public authenticated `/ready` returns `200`.

A persistent public DNS/transport failure is recorded as a **transport blocker**, not a model/runtime blocker. The notebook continues to evidence and cleanup instead of aborting.

---

### Tiếng Việt

Core production demo đã kiểm tra model thực tế, DFlash2, authenticated gateway, non-streaming generation và SSE thông qua loopback.

Bước này chỉ qualification **external transport**. DNS của Quick Tunnel có thể gặp lỗi tạm thời, vì vậy notebook sẽ retry DNS + HTTP trong tối đa `PUBLIC_DNS_RETRY_SECONDS` đồng thời kiểm tra lại:

- external exposure vẫn ở trạng thái `ready`;
- tunnel vẫn thuộc quyền quản lý của demo và còn hoạt động;
- public URL không thay đổi;
- request công khai tới `/v1/models` khi không xác thực trả về `401`;
- request công khai có xác thực tới `/ready` trả về `200`.

Nếu lỗi public DNS/transport kéo dài, notebook ghi nhận đó là **transport blocker**, không phải model/runtime blocker. Notebook vẫn tiếp tục ghi evidence và cleanup thay vì dừng ngay.


## 12. Qualify the optional public Quick Tunnel transport / Qualification transport Quick Tunnel công khai tùy chọn

**English:** Only after the local core succeeds, this cell stops the local proxy exposure and starts `EXPOSURE_MODE=quick` as a separate transport phase. It retries public DNS/HTTP checks and classifies persistent tunnel failures independently from model/runtime health. A core PASS with public failure remains a transport-only blocker; the model core is not invalidated.

**Tiếng Việt:** Chỉ sau khi local core thành công, cell này dừng local proxy exposure và khởi động `EXPOSURE_MODE=quick` như một phase transport tách biệt. Cell retry các kiểm tra DNS/HTTP công khai và phân loại lỗi tunnel kéo dài độc lập với sức khỏe model/runtime. Core PASS nhưng public FAIL vẫn chỉ là transport blocker; kết quả core model không bị vô hiệu hóa.


In [ ]:
PUBLIC_QUICK_TUNNEL = "NOT_RUN"
PUBLIC_FAILURE_CLASS = None
PUBLIC_LAST_ERROR = None
PUBLIC_DNS_ADDRESSES = []
PUBLIC_UNAUTH_MODELS_HTTP = None
PUBLIC_AUTH_READY_HTTP = None
PUBLIC_ATTEMPTS = 0
PUBLIC_QUICK_START_RC = None
PUBLIC_QUICK_START_OUTPUT_TAIL = None
PUBLIC_URL = None
PUBLIC_STOP_PROXY_RC = None

core_generation_ok = (
    (not RUN_NONSTREAM_DEMO or (
        LOCAL_NONSTREAM_STATUS == 200
        and not LOCAL_NONSTREAM_FAILURE_CAPTURED
    ))
    and
    (not RUN_SSE_DEMO or (
        LOCAL_SSE_STATUS == 200
        and LOCAL_SSE_DONE
        and not LOCAL_SSE_FAILURE_CAPTURED
    ))
)

if not core_generation_ok:
    PUBLIC_QUICK_TUNNEL = "NOT_RUN"
    PUBLIC_FAILURE_CLASS = "CORE_GENERATION_FAILURE"
    print("PUBLIC_QUICK_TUNNEL=NOT_RUN")
    print("PUBLIC_FAILURE_CLASS=CORE_GENERATION_FAILURE")
    print("PUBLIC_TRANSPORT_QUALIFICATION=SKIPPED_DUE_TO_CORE_FAILURE")

else:
    # Transition cleanly from proxy-only core mode to quick public mode.
    proxy_env, _ = build_canonical_demo_env(
        token,
        cloudflared,
        exposure_mode="proxy",
    )
    stop_proxy = run(
        ["./external.sh", "stop"],
        cwd=WORKDIR,
        env=proxy_env,
        check=False,
    )
    PUBLIC_STOP_PROXY_RC = stop_proxy.returncode
    print(f"PUBLIC_PHASE_PROXY_STOP_RC={PUBLIC_STOP_PROXY_RC}")

    quick_env, _ = build_canonical_demo_env(
        token,
        cloudflared,
        exposure_mode="quick",
    )

    quick_preflight = run(
        ["./external.sh", "preflight"],
        cwd=WORKDIR,
        env=quick_env,
        check=False,
    )
    print(f"PUBLIC_QUICK_PREFLIGHT_RC={quick_preflight.returncode}")

    quick_start = run(
        ["./external.sh", "start"],
        cwd=WORKDIR,
        env=quick_env,
        check=False,
    )
    PUBLIC_QUICK_START_RC = quick_start.returncode
    quick_output = quick_start.stdout or ""
    PUBLIC_QUICK_START_OUTPUT_TAIL = "\n".join(
        quick_output.splitlines()[-60:]
    )

    print(f"PUBLIC_QUICK_START_RC={PUBLIC_QUICK_START_RC}")

    if PUBLIC_QUICK_START_RC != 0:
        PUBLIC_QUICK_TUNNEL = "FAIL"

        if (
            "timed out waiting for authenticated public readiness" in quick_output
            and "LAST_UNAUTH_CODE=000" in quick_output
            and "LAST_AUTH_CODE=000" in quick_output
        ):
            PUBLIC_FAILURE_CLASS = "PUBLIC_READINESS_UNREACHABLE"
        elif "timed out waiting for authenticated public readiness" in quick_output:
            PUBLIC_FAILURE_CLASS = "PUBLIC_READINESS_TIMEOUT"
        elif "Quick Tunnel" in quick_output:
            PUBLIC_FAILURE_CLASS = "QUICK_TUNNEL_START_FAILURE"
        else:
            PUBLIC_FAILURE_CLASS = "PUBLIC_START_FAILURE"

        PUBLIC_LAST_ERROR = PUBLIC_QUICK_START_OUTPUT_TAIL

        print(f"PUBLIC_FAILURE_CLASS={PUBLIC_FAILURE_CLASS}")
        print("PUBLIC_TRANSPORT_BLOCKER=YES")
        print("MODEL_RUNTIME_BLOCKER=NO")
        if PUBLIC_LAST_ERROR:
            print("PUBLIC_QUICK_START_OUTPUT_TAIL_BEGIN")
            print(PUBLIC_LAST_ERROR)
            print("PUBLIC_QUICK_START_OUTPUT_TAIL_END")

        # Best-effort fail-closed exposure cleanup; backend remains owned/running.
        run(
            ["./external.sh", "stop"],
            cwd=WORKDIR,
            env=quick_env,
            check=False,
        )

    else:
        # Quick start proved the public auth path once. Re-verify the timeout
        # overlay on the newly launched proxy and then perform a bounded client
        # qualification of the public endpoint.
        verify_auth_proxy_timeout_process(
            AUTH_PROXY_BACKEND_TIMEOUT_SECONDS
        )

        PUBLIC_URL = run(
            ["./external.sh", "endpoint"],
            cwd=WORKDIR,
            env=quick_env,
        ).stdout.strip().rstrip("/")

        parsed_public = urllib.parse.urlsplit(PUBLIC_URL)
        assert parsed_public.scheme == "https" and parsed_public.hostname
        public_host = parsed_public.hostname

        deadline = time.monotonic() + float(PUBLIC_DNS_RETRY_SECONDS)

        while True:
            PUBLIC_ATTEMPTS += 1
            try:
                infos = socket.getaddrinfo(
                    public_host,
                    443,
                    type=socket.SOCK_STREAM,
                )
                PUBLIC_DNS_ADDRESSES = sorted({
                    item[4][0]
                    for item in infos
                    if item and len(item) >= 5 and item[4]
                })

                unauth = requests.get(
                    f"{PUBLIC_URL}/v1/models",
                    timeout=PUBLIC_HTTP_TIMEOUT_SECONDS,
                    headers={"Connection": "close"},
                )
                PUBLIC_UNAUTH_MODELS_HTTP = unauth.status_code

                auth_ready = requests.get(
                    f"{PUBLIC_URL}/ready",
                    headers={**AUTH, "Connection": "close"},
                    timeout=PUBLIC_HTTP_TIMEOUT_SECONDS,
                )
                PUBLIC_AUTH_READY_HTTP = auth_ready.status_code

                if (
                    PUBLIC_UNAUTH_MODELS_HTTP == 401
                    and PUBLIC_AUTH_READY_HTTP == 200
                ):
                    PUBLIC_QUICK_TUNNEL = "PASS"
                    PUBLIC_FAILURE_CLASS = None
                    PUBLIC_LAST_ERROR = None
                    break

                PUBLIC_FAILURE_CLASS = "PUBLIC_HTTP_CONTRACT"
                PUBLIC_LAST_ERROR = (
                    f"unauth_models={PUBLIC_UNAUTH_MODELS_HTTP} "
                    f"auth_ready={PUBLIC_AUTH_READY_HTTP}"
                )

            except socket.gaierror as exc:
                PUBLIC_FAILURE_CLASS = "DNS_RESOLUTION"
                PUBLIC_LAST_ERROR = f"{type(exc).__name__}: {exc}"
            except requests.exceptions.Timeout as exc:
                PUBLIC_FAILURE_CLASS = "HTTP_TIMEOUT"
                PUBLIC_LAST_ERROR = f"{type(exc).__name__}: {exc}"
            except requests.exceptions.ConnectionError as exc:
                text = str(exc)
                if (
                    "NameResolutionError" in text
                    or "Failed to resolve" in text
                    or "Name or service not known" in text
                ):
                    PUBLIC_FAILURE_CLASS = "DNS_RESOLUTION"
                else:
                    PUBLIC_FAILURE_CLASS = "CONNECTION_ERROR"
                PUBLIC_LAST_ERROR = f"{type(exc).__name__}: {exc}"
            except requests.RequestException as exc:
                PUBLIC_FAILURE_CLASS = "HTTP_REQUEST_ERROR"
                PUBLIC_LAST_ERROR = f"{type(exc).__name__}: {exc}"

            if time.monotonic() >= deadline:
                PUBLIC_QUICK_TUNNEL = "FAIL"
                break

            remaining = max(0.0, deadline - time.monotonic())
            sleep_for = min(float(PUBLIC_RETRY_INTERVAL_SECONDS), remaining)
            if sleep_for > 0:
                print(
                    f"PUBLIC_TRANSPORT_RETRY attempt={PUBLIC_ATTEMPTS} "
                    f"class={PUBLIC_FAILURE_CLASS} sleep={sleep_for:.1f}s"
                )
                time.sleep(sleep_for)

        print(f"PUBLIC_URL={PUBLIC_URL}")
        print(f"PUBLIC_QUICK_TUNNEL={PUBLIC_QUICK_TUNNEL}")
        print(f"PUBLIC_ATTEMPTS={PUBLIC_ATTEMPTS}")
        print(f"PUBLIC_DNS_ADDRESSES={PUBLIC_DNS_ADDRESSES}")
        print(f"PUBLIC_UNAUTH_V1_MODELS_HTTP={PUBLIC_UNAUTH_MODELS_HTTP}")
        print(f"PUBLIC_AUTH_READY_HTTP={PUBLIC_AUTH_READY_HTTP}")
        print(f"PUBLIC_FAILURE_CLASS={PUBLIC_FAILURE_CLASS}")

        if PUBLIC_LAST_ERROR:
            print(f"PUBLIC_LAST_ERROR={PUBLIC_LAST_ERROR[:1500]}")

        if PUBLIC_QUICK_TUNNEL == "PASS":
            print("PUBLIC_TRANSPORT_BLOCKER=NO")
        else:
            print("PUBLIC_TRANSPORT_BLOCKER=YES")
            print("MODEL_RUNTIME_BLOCKER=NO")

SESSION["public_quick_tunnel"] = PUBLIC_QUICK_TUNNEL
SESSION["public_failure_class"] = PUBLIC_FAILURE_CLASS
SESSION["public_quick_start_rc"] = PUBLIC_QUICK_START_RC
SESSION["public_proxy_stop_rc"] = PUBLIC_STOP_PROXY_RC
SESSION["public_attempts"] = PUBLIC_ATTEMPTS
SESSION["public_dns_address_count"] = len(PUBLIC_DNS_ADDRESSES)
SESSION["public_unauth_v1_models_http"] = PUBLIC_UNAUTH_MODELS_HTTP
SESSION["public_auth_ready_http"] = PUBLIC_AUTH_READY_HTTP
SESSION["public_endpoint_host"] = (
    urllib.parse.urlsplit(PUBLIC_URL).hostname
    if PUBLIC_URL
    else None
)


## Concurrency behavior / Hành vi concurrency

### English

`v1.0.0` intentionally has one active inference slot (`PARALLEL_SLOTS=1`). Overlapping inference is expected to return HTTP `429` with `muse_demo_busy`.

This production-demo notebook does **not** automatically generate overlapping real-model requests because doing so wastes GPU quota. The deterministic contract suite and publication evidence already cover admission behavior. The notebook focuses real GPU work on one local non-streaming generation and one local SSE generation.

---

### Tiếng Việt

`v1.0.0` chủ động chỉ sử dụng một inference slot hoạt động (`PARALLEL_SLOTS=1`). Khi các inference request chồng lấn nhau, hành vi mong đợi là trả về HTTP `429` với mã `muse_demo_busy`.

Production-demo notebook này **không** tự động tạo các real-model request chồng lấn vì việc đó làm lãng phí GPU quota. Deterministic contract suite và publication evidence đã bao phủ hành vi admission này. Notebook tập trung phần GPU thực tế vào một local non-streaming generation và một local SSE generation.


## 13. Write the secret-free production-demo evidence and final verdicts / Ghi evidence không chứa secret và các verdict cuối

**English:** This cell consolidates all prior gates into the evidence JSON at `/kaggle/working/muse-glimmer-30b-v1.0.0-demo-evidence.json`. Core PASS requires frozen identity, canonical environment, local auth, visible non-stream output, and visible SSE output. Public Quick Tunnel is evaluated separately. Important lines are `VISIBLE_ANSWER_GATE`, `CORE_PRODUCTION_DEMO`, `PUBLIC_QUICK_TUNNEL`, and `OVERALL_PRODUCTION_DEMO`.

**Tiếng Việt:** Cell này tổng hợp tất cả gate trước đó vào evidence JSON tại `/kaggle/working/muse-glimmer-30b-v1.0.0-demo-evidence.json`. Core PASS yêu cầu frozen identity, canonical environment, local auth, visible output ở non-stream và visible output ở SSE. Public Quick Tunnel được đánh giá riêng. Các dòng quan trọng là `VISIBLE_ANSWER_GATE`, `CORE_PRODUCTION_DEMO`, `PUBLIC_QUICK_TUNNEL` và `OVERALL_PRODUCTION_DEMO`.


In [ ]:
SESSION["completed_at_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()

SESSION["publication_identity_verified"] = (
    SESSION.get("actual_head") == EXPECTED_HEAD
    and SESSION.get("actual_tree") == EXPECTED_TREE
)

core_pass = all([
    SESSION.get("publication_identity_verified"),
    SESSION.get("canonical_environment") == "PASS",
    SESSION.get("demo_request_budget_gate") == "PASS",
    SESSION.get("demo_reasoning_strength") == "low",
    SESSION.get("auth_proxy_timeout_overlay") == "PASS",
    SESSION.get("core_exposure_mode") == "proxy",
    SESSION.get("auth_proxy_timeout_process_gate") == "PASS",
    SESSION.get("post_start_source_manifest") == "PASS",
    SESSION.get("rerun_recovery") == "PASS",
    SESSION.get("port_8088_free_before_start") is True,
    SESSION.get("port_8090_free_before_start") is True,
    SESSION.get("local_unauth_v1_models_http") == 401,
    SESSION.get("local_auth_ready_http") == 200,
    SESSION.get("local_auth_v1_models_http") == 200,
    (not RUN_NONSTREAM_DEMO) or (
        SESSION.get("local_nonstream_http") == 200
        and SESSION.get("local_nonstream_failure_captured") is False
        and (SESSION.get("local_nonstream_content_chars") or 0) > 0
        and SESSION.get("local_nonstream_visible_answer") is True
    ),
    (not RUN_SSE_DEMO) or (
        SESSION.get("local_sse_done") is True
        and SESSION.get("local_sse_failure_captured") is False
        and (SESSION.get("local_sse_content_chars") or 0) > 0
        and SESSION.get("local_sse_visible_answer") is True
    ),
])

public_pass = SESSION.get("public_quick_tunnel") == "PASS"

SESSION["visible_answer_gate"] = "PASS" if all([
    (not RUN_NONSTREAM_DEMO) or SESSION.get("local_nonstream_visible_answer") is True,
    (not RUN_SSE_DEMO) or SESSION.get("local_sse_visible_answer") is True,
]) else "FAIL"

SESSION["core_production_demo"] = "PASS" if core_pass else "FAIL"
SESSION["public_quick_tunnel_verdict"] = "PASS" if public_pass else "FAIL"
SESSION["model_runtime_blocker"] = "NO" if core_pass else "YES"
SESSION["transport_blocker"] = "NO" if public_pass else "YES"

if core_pass and public_pass:
    SESSION["overall_production_demo"] = "PASS"
elif core_pass:
    SESSION["overall_production_demo"] = "PARTIAL_TRANSPORT_BLOCKER"
else:
    SESSION["overall_production_demo"] = "FAIL"

for forbidden_key in ("token", "MUSE_API_TOKEN", "prompt", "completion"):
    assert forbidden_key not in SESSION

EVIDENCE_PATH.write_text(
    json.dumps(SESSION, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(EVIDENCE_PATH)
print(json.dumps(SESSION, indent=2, ensure_ascii=False))

print(f"VISIBLE_ANSWER_GATE={SESSION['visible_answer_gate']}")
print(f"CORE_PRODUCTION_DEMO={SESSION['core_production_demo']}")
print(f"PUBLIC_QUICK_TUNNEL={SESSION['public_quick_tunnel_verdict']}")
print(f"MODEL_RUNTIME_BLOCKER={SESSION['model_runtime_blocker']}")
print(f"TRANSPORT_BLOCKER={SESSION['transport_blocker']}")
print(f"OVERALL_PRODUCTION_DEMO={SESSION['overall_production_demo']}")

print("PRODUCTION_DEMO_EVIDENCE=PASS" if core_pass else "PRODUCTION_DEMO_EVIDENCE=FAIL_CAPTURED")


## Cleanup / Dọn dẹp

### English

Cleanup runs after evidence even when Quick Tunnel qualification ends with a transport blocker.

With the defaults:

- `AUTO_CLEANUP_AT_END=True` stops the tunnel/gateway first and then the owned backend.
- `AUTO_DELETE_GENERATED_TOKEN_AT_END=True` removes the notebook-generated token file.

A public DNS/transport failure therefore does not leave the demo intentionally running just because the public qualification did not pass.

---

### Tiếng Việt

Cleanup vẫn chạy sau khi evidence được ghi, kể cả khi qualification của Quick Tunnel kết thúc với một transport blocker.

Với cấu hình mặc định:

- `AUTO_CLEANUP_AT_END=True` dừng tunnel/gateway trước, sau đó dừng backend thuộc quyền quản lý của demo.
- `AUTO_DELETE_GENERATED_TOKEN_AT_END=True` xóa file token do notebook tự tạo.

Vì vậy, một lỗi public DNS/transport sẽ không khiến demo bị cố ý để tiếp tục chạy chỉ vì public qualification chưa đạt.


## 14. Clean up owned processes and report only current-run forensics / Dọn dẹp process thuộc demo và chỉ báo forensic của run hiện tại

**English:** This final cell runs canonical `stop-all`, deletes any notebook-generated token file when configured, updates the evidence JSON with cleanup state, and asserts only real cleanup failure. A semantic FAIL no longer turns the cleanup cell red. The cell reports a forensic ZIP only when `FORENSIC_CAPTURE_COUNT` for this run is greater than zero; otherwise it prints `FINAL_FORENSIC_ZIP=NOT_CREATED_THIS_RUN`. It then prints `FINAL_NOTEBOOK_RESULT=PASS`, `PARTIAL_TRANSPORT_BLOCKER`, or `FAIL`, followed by `NOTEBOOK_EXECUTION_COMPLETED=PASS`.

**Tiếng Việt:** Cell cuối này chạy canonical `stop-all`, xóa token file do notebook tự tạo khi được cấu hình, cập nhật evidence JSON với trạng thái cleanup và chỉ assert khi cleanup thực sự thất bại. Semantic FAIL không còn làm cleanup cell đỏ. Cell chỉ báo forensic ZIP khi `FORENSIC_CAPTURE_COUNT` của chính run này lớn hơn 0; nếu không sẽ in `FINAL_FORENSIC_ZIP=NOT_CREATED_THIS_RUN`. Sau đó notebook in `FINAL_NOTEBOOK_RESULT=PASS`, `PARTIAL_TRANSPORT_BLOCKER` hoặc `FAIL`, rồi `NOTEBOOK_EXECUTION_COMPLETED=PASS`.


In [ ]:
generated_token_deleted = False
stop_rc = None

if AUTO_CLEANUP_AT_END:
    print("Stopping public exposure + owned backend...")

    try:
        stop_result = run(
            ["./external.sh", "stop-all"],
            cwd=WORKDIR,
            env=build_canonical_demo_env(
                token, cloudflared, exposure_mode="quick"
            )[0],
            check=False,
        )
        stop_rc = stop_result.returncode

        final_status = run(
            ["./external.sh", "status", "--format", "env"],
            cwd=WORKDIR,
            env=build_canonical_demo_env(
                token, cloudflared, exposure_mode="quick"
            )[0],
            check=False,
        ).stdout
    finally:
        if AUTO_DELETE_GENERATED_TOKEN_AT_END and GENERATED_TOKEN_FILE.exists():
            GENERATED_TOKEN_FILE.unlink()
            generated_token_deleted = True
            try:
                if GENERATED_SECRET_DIR.exists() and not any(GENERATED_SECRET_DIR.iterdir()):
                    GENERATED_SECRET_DIR.rmdir()
            except OSError:
                pass

    SESSION["cleanup_stop_rc"] = stop_rc
    SESSION["generated_token_deleted_at_cleanup"] = generated_token_deleted

    if EVIDENCE_PATH.exists():
        EVIDENCE_PATH.write_text(
            json.dumps(SESSION, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )

    if generated_token_deleted:
        print(f"GENERATED_TOKEN_FILE_DELETED={GENERATED_TOKEN_FILE}")
    elif SESSION.get("secret_source") == "auto_generated_file":
        print("WARNING: Generated token file was expected but was already absent.")
    else:
        print("GENERATED_TOKEN_FILE=NOT_CREATED_BY_NOTEBOOK")

    assert stop_rc == 0, f"./external.sh stop-all failed with rc={stop_rc}"
    print("AUTO_CLEANUP=PASS")
    print("GENERATED_TOKEN_CLEANUP=PASS")
else:
    SESSION["cleanup_stop_rc"] = None
    SESSION["generated_token_deleted_at_cleanup"] = False

    print("AUTO_CLEANUP=SKIPPED")
    print("The temporary public endpoint is still operator-controlled.")
    print("When finished, run:")
    print("  ./external.sh stop-all")

    if SESSION.get("secret_source") == "auto_generated_file":
        print("WARNING: The generated token file remains on disk while the demo is running.")
        print(f"TOKEN_FILE={GENERATED_TOKEN_FILE}")
        print("IMPORTANT: Delete it after stopping the demo.")

print(f"FINAL_CORE_PRODUCTION_DEMO={SESSION.get('core_production_demo', 'UNKNOWN')}")
print(f"FINAL_PUBLIC_QUICK_TUNNEL={SESSION.get('public_quick_tunnel_verdict', 'UNKNOWN')}")
print(f"FINAL_TRANSPORT_BLOCKER={SESSION.get('transport_blocker', 'UNKNOWN')}")


core_verdict = SESSION.get("core_production_demo", "UNKNOWN")
public_verdict = SESSION.get("public_quick_tunnel_verdict", "UNKNOWN")
overall_verdict = SESSION.get("overall_production_demo", "UNKNOWN")

if overall_verdict == "PASS":
    final_notebook_result = "PASS"
elif overall_verdict == "PARTIAL_TRANSPORT_BLOCKER":
    final_notebook_result = "PARTIAL_TRANSPORT_BLOCKER"
else:
    final_notebook_result = "FAIL"

print(f"FINAL_NOTEBOOK_RESULT={final_notebook_result}")
print(f"FINAL_OVERALL_PRODUCTION_DEMO={overall_verdict}")
print(f"FINAL_EVIDENCE_PATH={EVIDENCE_PATH}")
forensic_capture_count = int(SESSION.get("forensic_capture_count", 0))
print(f"FORENSIC_CAPTURE_COUNT={forensic_capture_count}")
if forensic_capture_count > 0 and FORENSIC_ZIP_PATH.exists():
    print(f"FINAL_FORENSIC_ZIP={FORENSIC_ZIP_PATH}")
else:
    print("FINAL_FORENSIC_ZIP=NOT_CREATED_THIS_RUN")

print("NOTEBOOK_EXECUTION_COMPLETED=PASS")
